# Global Automotive Investment Database — Block 8

## Australia / ASX filings and deterministic fundamentals extraction

This notebook builds the Australian component of the Global Automotive Investment Database.

It:

1. identifies Australian securities in the point-in-time automotive ETF universe;
2. resolves ASX-listed issuers and filing relationships;
3. acquires official ASX disclosures and annual reports;
4. performs deterministic document extraction and account mapping;
5. standardises accounting facts to the global canonical schema;
6. emits explicit review queues for Block 9 — AI Enrichment and Quality Control.

Block 8 does not use an LLM, a vision-language model or an external AI document service.

### Upstream dependencies

- Block 2 — Security Master
- Block 3 — USA / SEC fundamentals
- Block 4 — Europe / ESMA fundamentals and canonical concept schema
- Block 5 — Japan / EDINET fundamentals
- Block 6 — Korea / DART fundamentals
- Block 7 — Hong Kong / HKEX and China / CNINFO

### Downstream dependencies

- Block 9 — AI-Assisted Quality Control
- Block 10 — Global Fundamentals


## Notebook identity

This notebook collects Australian financial reports from issuer
investor-relations websites.

It does not use the ASX historical-announcement search endpoint. Each issuer has
configured investor-relations landing pages. The crawler follows report-related
links within the issuer domain, identifies annual and half-year financial-report
PDFs, downloads them, and preserves publication dates where available.

In [1]:
# 1. IMPORTS

!pip -q install pandas pyarrow pypdf tqdm requests beautifulsoup4 lxml

import gc
import hashlib
import json
import re
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import requests

from bs4 import BeautifulSoup

from pypdf import PdfReader
from tqdm.auto import tqdm

print("Imports complete.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 4.9 MB/s eta 0:00:00
Imports complete.


In [2]:
# 2. PROJECT PATHS AND SETTINGS

USE_GOOGLE_DRIVE = True

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

    PROJECT_ROOT = Path(
        "/content/drive/MyDrive/Colab Notebooks/00 A1 Auto Factor Strategy"
    )
else:
    PROJECT_ROOT = Path(
        "/content/global_automotive_investment_database"
    )

DATA_ROOT = PROJECT_ROOT / "data"
INTERIM_ROOT = DATA_ROOT / "interim"
EXTERNAL_ROOT = DATA_ROOT / "external"

BLOCK_2_OUTPUT_DIR = INTERIM_ROOT / "block_2"
BLOCK_7_OUTPUT_DIR = INTERIM_ROOT / "block_7"
BLOCK_8_OUTPUT_DIR = INTERIM_ROOT / "block_8"
BLOCK_9_OUTPUT_DIR = INTERIM_ROOT / "block_9"
BLOCK_9_MANIFEST_PATH = (
    BLOCK_9_OUTPUT_DIR / "block_9_manifest.json"
)
BLOCK_8_MANIFEST_PATH = (
    BLOCK_8_OUTPUT_DIR / "block_8_manifest.json"
)

ASX_ROOT = EXTERNAL_ROOT / "asx"
ASX_DOCUMENT_DIR = ASX_ROOT / "documents"
ASX_METADATA_PATH = (
    ASX_ROOT / "asx_announcements_metadata.csv"
)
ASX_TEXT_CACHE_DIR = (
    BLOCK_8_OUTPUT_DIR / "text_cache"
)
ASX_RESPONSE_CACHE_DIR = (
    BLOCK_8_OUTPUT_DIR / "asx_response_cache"
)

for path in [
    BLOCK_8_OUTPUT_DIR,
    ASX_ROOT,
    ASX_DOCUMENT_DIR,
    ASX_TEXT_CACHE_DIR,
    ASX_RESPONSE_CACHE_DIR,
]:
    path.mkdir(parents=True, exist_ok=True)

TARGET_ETFS = {
    "DRIV",
    "CARZ",
    "IDRV",
    "KARS",
}

MAX_DOCUMENTS_TO_PARSE = None
MAX_PDF_PAGES = None
RECURSIVE_PDF_DISCOVERY = True
ALLOW_FILE_MTIME_FALLBACK = True

REQUEST_TIMEOUT_SECONDS = 60
MAX_RETRIES = 4
MAX_REPORTS_PER_SECURITY = None

IR_CRAWL_MAX_DEPTH = 1
IR_CRAWL_MAX_PAGES_PER_ISSUER = 20
IR_LINK_REQUEST_INTERVAL_SECONDS = 0.30
ALLOW_LOCAL_PDF_FALLBACK = True

SEC_SUBMISSIONS_BASE_URL = (
    "https://data.sec.gov/submissions"
)
SEC_ARCHIVES_BASE_URL = (
    "https://www.sec.gov/Archives/edgar/data"
)
SEC_MAX_FILINGS_PER_ISSUER = 20

SEC_INCLUDE_FORMS = {
    "INR": {
        "20-F",
        "20-F/A",
        "6-K",
    },
    "LTM": {
        "10-K",
        "10-K/A",
        "10-Q",
        "20-F",
        "6-K",
    },
}

SEC_CIK_BY_ASX_CODE = {
    "INR": "0001896084",
    "LTM": "0001977303",
}

PERSIST_BLOCK_8_OUTPUTS = True
OVERWRITE_PERSISTED_OUTPUTS = True
APPLY_BLOCK_9_SYNONYM_REGISTRY = True

print("Project root:", PROJECT_ROOT)
print(
    "ASX document directory:",
    ASX_DOCUMENT_DIR,
)
print(
    "Block 8 output directory:",
    BLOCK_8_OUTPUT_DIR,
)

Mounted at /content/drive
Project root: /content/drive/MyDrive/Colab Notebooks/00 A1 Auto Factor Strategy
ASX document directory: /content/drive/MyDrive/Colab Notebooks/00 A1 Auto Factor Strategy/data/external/asx/documents
Block 8 output directory: /content/drive/MyDrive/Colab Notebooks/00 A1 Auto Factor Strategy/data/interim/block_8


In [3]:
# 3. LOAD THE MINIMUM REQUIRED UPSTREAM TABLES

def load_manifest_table(output_dir, table_name):
    output_dir = Path(output_dir)
    manifest_path = output_dir / f"{output_dir.name}_manifest.json"

    if not manifest_path.exists():
        raise FileNotFoundError(
            f"Manifest not found: {manifest_path}"
        )

    with manifest_path.open("r", encoding="utf-8") as file:
        manifest = json.load(file)

    records = {
        item["table_name"]: item
        for item in manifest.get("tables", [])
    }

    if table_name not in records:
        raise RuntimeError(
            f"{manifest_path.name} does not contain {table_name}."
        )

    table_path = Path(records[table_name]["path"])

    if not table_path.exists():
        raise FileNotFoundError(table_path)

    return pd.read_parquet(table_path)


security_master_df = load_manifest_table(
    BLOCK_2_OUTPUT_DIR,
    "security_master_df",
)

membership_intervals_df = load_manifest_table(
    BLOCK_2_OUTPUT_DIR,
    "security_etf_membership_intervals_df",
)


def load_shared_dictionary():
    candidates = [
        "china_standard_concept_dictionary_df",
        "mainland_china_standard_concept_dictionary_df",
    ]

    manifest_path = (
        BLOCK_7_OUTPUT_DIR / "block_7_manifest.json"
    )

    if not manifest_path.exists():
        return pd.DataFrame()

    with manifest_path.open("r", encoding="utf-8") as file:
        manifest = json.load(file)

    records = {
        item["table_name"]: item
        for item in manifest.get("tables", [])
    }

    for table_name in candidates:
        if table_name in records:
            path = Path(records[table_name]["path"])

            if path.exists():
                return pd.read_parquet(path)

    return pd.DataFrame()


shared_concept_dictionary_df = load_shared_dictionary()


# ------------------------------------------------------------
# Load the cumulative Block 9 accepted-synonym registry
# ------------------------------------------------------------

def load_block9_synonym_registry(manifest_path):
    """Load Block 9's cumulative accepted synonym registry.

    Block 9 uses a persistence manifest whose records may be stored under
    either ``outputs`` or ``tables`` and may use ``saved_file`` or ``path``.
    The feedback loop remains optional so Block 8 can still run before
    Block 9 has been created.
    """
    expected_columns = [
        "normalised_source_account_label",
        "standard_concept",
        "statement_type",
        "reporting_scope",
        "accepted_observations",
        "unique_issuers",
        "unique_filings",
        "first_accepted_at_utc",
        "last_accepted_at_utc",
        "registry_generation",
    ]

    empty = pd.DataFrame(columns=expected_columns)
    manifest_path = Path(manifest_path)

    if not APPLY_BLOCK_9_SYNONYM_REGISTRY:
        return empty, {
            "status": "DISABLED",
            "manifest_path": str(manifest_path),
            "loaded_table_name": None,
            "loaded_rows": 0,
            "error": pd.NA,
        }

    if not manifest_path.exists():
        return empty, {
            "status": "MANIFEST_MISSING",
            "manifest_path": str(manifest_path),
            "loaded_table_name": None,
            "loaded_rows": 0,
            "error": pd.NA,
        }

    try:
        payload = json.loads(
            manifest_path.read_text(encoding="utf-8")
        )
    except Exception as exc:
        return empty, {
            "status": "MANIFEST_READ_FAILED",
            "manifest_path": str(manifest_path),
            "loaded_table_name": None,
            "loaded_rows": 0,
            "error": repr(exc),
        }

    records = []
    for key in ["outputs", "tables"]:
        value = payload.get(key)
        if isinstance(value, list):
            records.extend(value)

    aliases = {
        "accepted_synonym_registry",
        "accepted_synonym_registry_df",
    }

    candidates = []
    for record in records:
        if not isinstance(record, dict):
            continue

        table_name = str(
            record.get(
                "table_name",
                record.get("name", ""),
            )
        ).strip()

        if table_name not in aliases:
            continue

        saved_file = (
            record.get("saved_file")
            or record.get("path")
            or record.get("file_path")
            or record.get("output_path")
        )

        if not saved_file:
            continue

        candidates.append({
            "table_name": table_name,
            "file_type": str(
                record.get("file_type", "")
            ).lower(),
            "path": Path(saved_file),
        })

    candidates = sorted(
        candidates,
        key=lambda item: (
            item["file_type"] != "parquet",
            item["table_name"] != "accepted_synonym_registry",
        ),
    )

    if not candidates:
        for filename in [
            "accepted_synonym_registry.parquet",
            "accepted_synonym_registry_df.parquet",
            "accepted_synonym_registry.csv",
            "accepted_synonym_registry_df.csv",
        ]:
            candidate_path = manifest_path.parent / filename
            if candidate_path.exists():
                candidates.append({
                    "table_name": candidate_path.stem,
                    "file_type": candidate_path.suffix.lstrip(".").lower(),
                    "path": candidate_path,
                })

    if not candidates:
        return empty, {
            "status": "TABLE_NOT_FOUND",
            "manifest_path": str(manifest_path),
            "loaded_table_name": None,
            "loaded_rows": 0,
            "error": pd.NA,
        }

    selected = candidates[0]

    if not selected["path"].exists():
        return empty, {
            "status": "FILE_NOT_FOUND",
            "manifest_path": str(manifest_path),
            "loaded_table_name": selected["table_name"],
            "loaded_rows": 0,
            "error": str(selected["path"]),
        }

    try:
        if (
            selected["file_type"] == "csv"
            or selected["path"].suffix.lower() == ".csv"
        ):
            registry = pd.read_csv(
                selected["path"],
                low_memory=False,
            )
        else:
            registry = pd.read_parquet(selected["path"])
    except Exception as exc:
        return empty, {
            "status": "LOAD_FAILED",
            "manifest_path": str(manifest_path),
            "loaded_table_name": selected["table_name"],
            "loaded_rows": 0,
            "error": repr(exc),
        }

    for column in expected_columns:
        if column not in registry.columns:
            registry[column] = pd.NA

    registry = registry[expected_columns].copy()

    return registry, {
        "status": "LOADED",
        "manifest_path": str(manifest_path),
        "loaded_table_name": selected["table_name"],
        "loaded_rows": int(len(registry)),
        "error": pd.NA,
    }


(
    block9_accepted_synonym_registry_raw_df,
    block9_synonym_registry_load_status,
) = load_block9_synonym_registry(
    BLOCK_9_MANIFEST_PATH
)

print("Security-master rows:", len(security_master_df))
print("Membership rows:", len(membership_intervals_df))
print(
    "Shared canonical concepts:",
    len(shared_concept_dictionary_df),
)
print(
    "Block 9 synonym-registry status:",
    block9_synonym_registry_load_status["status"],
)
print(
    "Block 9 synonym-registry rows loaded:",
    block9_synonym_registry_load_status["loaded_rows"],
)

Security-master rows: 512
Membership rows: 8028
Shared canonical concepts: 96
Block 9 synonym-registry status: LOADED
Block 9 synonym-registry rows loaded: 531


In [4]:
# 4. BUILD THE AUSTRALIAN INVESTABLE UNIVERSE

def normalise_asx_code(value):
    if pd.isna(value):
        return pd.NA

    text = str(value).upper().strip()
    text = re.sub(r"\.(AX|AU)$", "", text)
    text = re.sub(r"[^A-Z0-9]", "", text)

    return text if text else pd.NA


etf_column = next(
    (
        column
        for column in [
            "etf_ticker",
            "etf",
            "fund_ticker",
            "fund",
        ]
        if column in membership_intervals_df.columns
    ),
    None,
)

if etf_column is None:
    raise RuntimeError(
        "No ETF identifier column was found in the membership table."
    )

membership_intervals_df = membership_intervals_df.copy()
membership_intervals_df["etf_ticker"] = (
    membership_intervals_df[etf_column]
    .astype("string")
    .str.upper()
    .str.strip()
)

target_memberships_df = (
    membership_intervals_df[
        membership_intervals_df[
            "etf_ticker"
        ].isin(TARGET_ETFS)
    ]
    .copy()
)

target_security_ids = set(
    target_memberships_df[
        "security_id"
    ].dropna()
)

target_security_master_df = (
    security_master_df[
        security_master_df[
            "security_id"
        ].isin(target_security_ids)
    ]
    .copy()
)

for column in [
    "ticker",
    "listing_ticker",
    "exchange",
    "mic",
    "country",
    "listing_country",
    "issuer_country",
    "issuer_name",
]:
    if column not in target_security_master_df.columns:
        target_security_master_df[column] = pd.NA

target_security_master_df["asx_code"] = (
    target_security_master_df[
        "listing_ticker"
    ].map(normalise_asx_code)
    .combine_first(
        target_security_master_df[
            "ticker"
        ].map(normalise_asx_code)
    )
)

is_asx = (
    target_security_master_df[
        "mic"
    ].astype("string").eq("XASX")
    | target_security_master_df[
        "exchange"
    ].astype("string").str.upper().isin(
        {"ASX", "XASX"}
    )
    | (
        target_security_master_df[
            "listing_country"
        ].astype("string").str.upper().isin(
            {"AU", "AUS", "AUSTRALIA"}
        )
        & target_security_master_df[
            "asx_code"
        ].notna()
    )
)

australia_security_universe_df = (
    target_security_master_df[
        is_asx.fillna(False)
    ]
    .drop_duplicates("security_id")
    .reset_index(drop=True)
)

australia_issuer_universe_df = (
    australia_security_universe_df
    .groupby("issuer_id", dropna=False)
    .agg(
        issuer_name=("issuer_name", "first"),
        security_count=("security_id", "nunique"),
        asx_codes=(
            "asx_code",
            lambda values: " | ".join(
                sorted(
                    set(
                        values.dropna().astype(str)
                    )
                )
            ),
        ),
    )
    .reset_index()
)

if australia_security_universe_df.empty:
    raise RuntimeError(
        "No ASX-listed securities were identified from Block 2."
    )

print(
    "Australian securities:",
    australia_security_universe_df[
        "security_id"
    ].nunique(),
)
print(
    "Australian issuers:",
    australia_issuer_universe_df[
        "issuer_id"
    ].nunique(),
)

display(
    australia_security_universe_df[
        [
            "security_id",
            "issuer_id",
            "issuer_name",
            "asx_code",
        ]
    ]
)

Australian securities: 7
Australian issuers: 7


,security_id,issuer_id,issuer_name,asx_code
0,GAS_3FA3D5C335E42157E108,GAI_B79EA297FBC86386AD61,LIONTOWN RESOURCES LIMITED,LTR
1,GAS_55366DA533A7D9FABEC4,GAI_C905CFCC658D674645F2,PILBARA MINERALS LIMITED,PLS
2,GAS_5E2CD5EFF6246DBE5C8B,GAI_FDF7FFF16173ED43E31A,Arcadium Lithium PLC,LTM
3,GAS_95701E92432FF7F04FAB,GAI_35C1E52F44016A33F594,ALLKEM LIMITED,AKE
4,GAS_C18E53D71BACF1162D03,GAI_B2E4ECD04E4E4CD53273,Nickel Industries Ltd,NIC
5,GAS_DF7F0C7BC341A58956DC,GAI_ECA2EA117981D9E0FF2B,IGO LIMITED,IGO
6,GAS_E1EFE9F013256A3706F0,GAI_2C802D47EF667C123F15,ioneer Ltd,INR


In [5]:
# 5. DISCOVER AND DOWNLOAD FINANCIAL REPORTS FROM ISSUER IR WEBSITES

import time
from collections import deque
from urllib.parse import urljoin, urlparse



known_asx_codes = sorted(
    set(
        australia_security_universe_df[
            "asx_code"
        ]
        .dropna()
        .astype(str)
        .str.upper()
        .str.strip()
    )
)

if not known_asx_codes:
    raise RuntimeError(
        "The Australian security universe contains no usable ASX codes."
    )

REPORT_TITLE_PATTERN = re.compile(
    r"(?:annual report|annual financial report|full year|full-year|"
    r"half year|half-year|half yearly|interim report|"
    r"interim financial report|appendix 4e|appendix 4d|"
    r"financial statements|financial report)",
    flags=re.IGNORECASE,
)

EXCLUDE_TITLE_PATTERN = re.compile(
    r"(?:presentation|webcast|transcript|quarterly|sustainability|"
    r"esg|modern slavery|corporate governance|appendix 4g|"
    r"notice of meeting|proxy|remuneration report|tax transparency)",
    flags=re.IGNORECASE,
)

CRAWL_LINK_PATTERN = re.compile(
    r"(?:investor|report|financial|announcement|filing|result|"
    r"annual|half-year|half year|interim)",
    flags=re.IGNORECASE,
)

PDF_HINT_PATTERN = re.compile(
    r"(?:\.pdf(?:$|\?)|download|document|attachment)",
    flags=re.IGNORECASE,
)


ISSUER_IR_SOURCES = {
    "IGO": [
        "https://www.igo.com.au/site/investor-center/annual-reports",
        "https://www.igo.com.au/site/investor-center/financial-reports",
    ],
    "PLS": [
        "https://pilbaraminerals.com.au/investors/reports-and-asx-announcements/",
        "https://www.pilbaraminerals.com.au/investors/reports-and-asx-announcements/",
        "https://pilbaraminerals.com.au/sustainability/reporting-and-disclosures/",
        "https://pilbaraminerals.com.au/news-stories/fy24-full-year-results/",
    ],
    "LTR": [
        "https://www.liontown.com/investors/reports-presentations/",
        "https://www.liontown.com/investors/annual-reporting/",
    ],
    "NIC": [
        "https://nickelindustries.com/investor-centre/company-reports/",
        "https://nickelindustries.com/investor-centre/asx-announcements/",
    ],
    "INR": [
        "https://www.ioneer.com/investors/",
        "https://www.ioneer.com/investors/presentations/",
        "https://www.ioneer.com/about/corporate-directory/",
    ],
    "LTM": [
        "https://ir.arcadiumlithium.com/investors/financials-and-filings/",
        "https://ir.arcadiumlithium.com/investors/financials-and-filings/filings/",
    ],
    "AKE": [
        "https://www.allkem.co/investors/reports-and-presentations/",
        "https://www.allkem.co/investors/asx-announcements/",
    ],
}


DISCOVERY_COLUMNS = [
    "document_id",
    "asx_code",
    "issuer_name",
    "headline",
    "document_url",
    "source_page_url",
    "publication_datetime_raw",
    "available_datetime",
    "document_type",
    "document_priority",
    "discovery_method",
    "source_system",
]

CRAWL_LOG_COLUMNS = [
    "asx_code",
    "issuer_name",
    "page_url",
    "crawl_depth",
    "status",
    "http_status",
    "content_type",
    "response_bytes",
    "report_links_found",
    "error",
]

DOWNLOAD_COLUMNS = [
    "document_id",
    "asx_code",
    "issuer_name",
    "headline",
    "document_url",
    "source_page_url",
    "available_datetime",
    "document_type",
    "document_priority",
    "document_filename",
    "document_path",
    "document_format",
    "document_exists",
    "download_status",
    "download_error",
    "source_system",
]


SESSION = requests.Session()
SESSION.headers.update({
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0 Safari/537.36"
    ),
    "Accept": (
        "text/html,application/xhtml+xml,application/xml;q=0.9,"
        "application/pdf;q=0.8,*/*;q=0.7"
    ),
    "Accept-Language": "en-AU,en;q=0.9",
})




def compute_file_sha256(path):
    digest = hashlib.sha256()

    with Path(path).open("rb") as file:
        for chunk in iter(
            lambda: file.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()

def request_with_retries(url):
    last_error = None

    request_variants = [
        {"verify": True},
        {"verify": False},
    ]

    for request_options in request_variants:
        for attempt in range(
            1,
            MAX_RETRIES + 1,
        ):
            try:
                response = SESSION.get(
                    url,
                    timeout=REQUEST_TIMEOUT_SECONDS,
                    allow_redirects=True,
                    verify=request_options[
                        "verify"
                    ],
                )
                response.raise_for_status()
                return response

            except Exception as exc:
                last_error = exc
                time.sleep(
                    min(2 ** attempt, 12)
                )

    raise RuntimeError(
        f"Request failed for {url}: "
        f"{last_error!r}"
    )


def normalise_domain(url):
    hostname = (
        urlparse(url).hostname
        or ""
    ).lower()

    return hostname.removeprefix("www.")


def is_same_issuer_domain(url, root_domains):
    domain = normalise_domain(url)

    return any(
        domain == root
        or domain.endswith("." + root)
        for root in root_domains
    )


def clean_link_text(value):
    return re.sub(
        r"\s+",
        " ",
        str(value),
    ).strip()


def infer_document_type(title, url):
    text = f"{title} {url}".lower()

    if re.search(
        r"(?:half year|half-year|half yearly|"
        r"interim report|interim financial|appendix 4d)",
        text,
    ):
        return "HALF_YEAR_REPORT", 2

    if re.search(
        r"(?:annual report|annual financial|full year|"
        r"full-year|appendix 4e)",
        text,
    ):
        return "ANNUAL_REPORT", 1

    return "OTHER_FINANCIAL_REPORT", 3


def extract_publication_datetime(context_text, url):
    text = f"{context_text} {url}"

    patterns = [
        (
            r"(?<!\d)(\d{1,2})[/-](\d{1,2})[/-](20\d{2})(?!\d)",
            "DMY",
        ),
        (
            r"(?<!\d)(20\d{2})[/-](\d{1,2})[/-](\d{1,2})(?!\d)",
            "YMD",
        ),
        (
            r"(?<!\d)(\d{1,2})\s+"
            r"(Jan(?:uary)?|Feb(?:ruary)?|Mar(?:ch)?|Apr(?:il)?|"
            r"May|Jun(?:e)?|Jul(?:y)?|Aug(?:ust)?|Sep(?:tember)?|"
            r"Oct(?:ober)?|Nov(?:ember)?|Dec(?:ember)?)\s+"
            r"(20\d{2})(?!\d)",
            "DMonY",
        ),
    ]

    for pattern, order in patterns:
        match = re.search(
            pattern,
            text,
            flags=re.IGNORECASE,
        )

        if not match:
            continue

        if order == "DMY":
            day, month, year = map(int, match.groups())
            timestamp = pd.Timestamp(
                year=year,
                month=month,
                day=day,
                hour=9,
            )

        elif order == "YMD":
            year, month, day = map(int, match.groups())
            timestamp = pd.Timestamp(
                year=year,
                month=month,
                day=day,
                hour=9,
            )

        else:
            timestamp = pd.to_datetime(
                " ".join(match.groups()),
                dayfirst=True,
                errors="coerce",
            )

            if pd.isna(timestamp):
                continue

            timestamp = pd.Timestamp(timestamp).replace(
                hour=9,
            )

        try:
            return timestamp.tz_localize(
                "Australia/Sydney",
                ambiguous="NaT",
                nonexistent="shift_forward",
            ).tz_convert("UTC")

        except Exception:
            return pd.to_datetime(
                timestamp,
                utc=True,
                errors="coerce",
            )

    return pd.NaT


def extract_report_links_from_page(
    page_url,
    html_content,
    asx_code,
    issuer_name,
):
    soup = BeautifulSoup(
        html_content,
        "lxml",
    )

    rows = []

    for link in soup.find_all(
        "a",
        href=True,
    ):
        href = clean_link_text(
            link.get("href", "")
        )

        if not href:
            continue

        absolute_url = urljoin(
            page_url,
            href,
        )

        link_text = clean_link_text(
            link.get_text(
                " ",
                strip=True,
            )
        )

        container = link.find_parent(
            ["li", "article", "tr", "div"]
        )

        context_text = clean_link_text(
            container.get_text(
                " ",
                strip=True,
            )
            if container is not None
            else link_text
        )

        combined_text = (
            f"{link_text} {context_text} "
            f"{absolute_url}"
        )

        if not REPORT_TITLE_PATTERN.search(
            combined_text
        ):
            continue

        if EXCLUDE_TITLE_PATTERN.search(
            combined_text
        ):
            continue

        if not PDF_HINT_PATTERN.search(
            absolute_url
        ) and len(link_text) < 4:
            continue

        headline = (
            link_text
            if len(link_text) >= 4
            else context_text
        )

        document_type, priority = (
            infer_document_type(
                headline,
                absolute_url,
            )
        )

        available_datetime = (
            extract_publication_datetime(
                context_text,
                absolute_url,
            )
        )

        document_id = hashlib.sha256(
            absolute_url.encode()
        ).hexdigest()[:24]

        rows.append({
            "document_id": document_id,
            "asx_code": asx_code,
            "issuer_name": issuer_name,
            "headline": headline[:500],
            "document_url": absolute_url,
            "source_page_url": page_url,
            "publication_datetime_raw": context_text[:1000],
            "available_datetime": available_datetime,
            "document_type": document_type,
            "document_priority": priority,
            "discovery_method": "ISSUER_IR_PAGE_LINK",
            "source_system": "ISSUER_INVESTOR_RELATIONS",
        })

    return pd.DataFrame(
        rows,
        columns=DISCOVERY_COLUMNS,
    )


def extract_crawl_links(
    page_url,
    html_content,
    root_domains,
):
    soup = BeautifulSoup(
        html_content,
        "lxml",
    )

    links = []

    for link in soup.find_all(
        "a",
        href=True,
    ):
        href = clean_link_text(
            link.get("href", "")
        )

        if not href:
            continue

        absolute_url = urljoin(
            page_url,
            href,
        )

        text = (
            clean_link_text(
                link.get_text(
                    " ",
                    strip=True,
                )
            )
            + " "
            + absolute_url
        )

        if not is_same_issuer_domain(
            absolute_url,
            root_domains,
        ):
            continue

        if not CRAWL_LINK_PATTERN.search(
            text
        ):
            continue

        if PDF_HINT_PATTERN.search(
            absolute_url
        ):
            continue

        links.append(absolute_url)

    return list(dict.fromkeys(links))


issuer_name_lookup = (
    australia_security_universe_df[
        [
            "asx_code",
            "issuer_name",
        ]
    ]
    .drop_duplicates("asx_code")
    .set_index("asx_code")[
        "issuer_name"
    ]
    .to_dict()
)



configured_asx_codes = sorted(
    set(ISSUER_IR_SOURCES)
)

unconfigured_asx_codes = sorted(
    set(known_asx_codes)
    .difference(configured_asx_codes)
)

print(
    "ASX codes passed to issuer IR crawler:",
    known_asx_codes,
)

if unconfigured_asx_codes:
    print(
        "Warning: no issuer IR source is configured for:",
        unconfigured_asx_codes,
    )

discovery_frames = []
crawl_log_rows = []

for asx_code in tqdm(
    known_asx_codes,
    desc="Crawling issuer IR websites",
):
    issuer_name = issuer_name_lookup.get(
        asx_code,
        asx_code,
    )

    seed_urls = ISSUER_IR_SOURCES.get(
        asx_code,
        [],
    )

    if not seed_urls:
        crawl_log_rows.append({
            "asx_code": asx_code,
            "issuer_name": issuer_name,
            "page_url": pd.NA,
            "crawl_depth": pd.NA,
            "status": "NO_IR_SOURCE_CONFIGURED",
            "http_status": pd.NA,
            "content_type": pd.NA,
            "response_bytes": 0,
            "report_links_found": 0,
            "error": pd.NA,
        })
        continue

    root_domains = {
        normalise_domain(url)
        for url in seed_urls
    }

    queue = deque(
        (url, 0)
        for url in seed_urls
    )

    visited = set()
    pages_processed = 0

    while (
        queue
        and pages_processed
        < IR_CRAWL_MAX_PAGES_PER_ISSUER
    ):
        page_url, depth = queue.popleft()

        if page_url in visited:
            continue

        visited.add(page_url)
        pages_processed += 1

        try:
            response = request_with_retries(
                page_url
            )

            content_type = (
                response.headers.get(
                    "Content-Type",
                    "",
                )
            )

            if "html" not in content_type.lower():
                crawl_log_rows.append({
                    "asx_code": asx_code,
                    "issuer_name": issuer_name,
                    "page_url": page_url,
                    "crawl_depth": depth,
                    "status": "NON_HTML_PAGE",
                    "http_status": response.status_code,
                    "content_type": content_type,
                    "response_bytes": len(response.content),
                    "report_links_found": 0,
                    "error": pd.NA,
                })
                continue

            report_frame = (
                extract_report_links_from_page(
                    response.url,
                    response.content,
                    asx_code,
                    issuer_name,
                )
            )

            if not report_frame.empty:
                discovery_frames.append(
                    report_frame
                )

            crawl_log_rows.append({
                "asx_code": asx_code,
                "issuer_name": issuer_name,
                "page_url": response.url,
                "crawl_depth": depth,
                "status": "PARSED",
                "http_status": response.status_code,
                "content_type": content_type,
                "response_bytes": len(response.content),
                "report_links_found": len(
                    report_frame
                ),
                "error": pd.NA,
            })

            if depth < IR_CRAWL_MAX_DEPTH:
                child_links = (
                    extract_crawl_links(
                        response.url,
                        response.content,
                        root_domains,
                    )
                )

                for child_url in child_links:
                    if child_url not in visited:
                        queue.append(
                            (
                                child_url,
                                depth + 1,
                            )
                        )

            time.sleep(
                IR_LINK_REQUEST_INTERVAL_SECONDS
            )

        except Exception as exc:
            crawl_log_rows.append({
                "asx_code": asx_code,
                "issuer_name": issuer_name,
                "page_url": page_url,
                "crawl_depth": depth,
                "status": "FAILED",
                "http_status": pd.NA,
                "content_type": pd.NA,
                "response_bytes": 0,
                "report_links_found": 0,
                "error": repr(exc),
            })


issuer_ir_crawl_log_df = pd.DataFrame(
    crawl_log_rows,
    columns=CRAWL_LOG_COLUMNS,
)

issuer_ir_reports_discovered_df = (
    pd.concat(
        discovery_frames,
        ignore_index=True,
        sort=False,
    )
    if discovery_frames
    else pd.DataFrame(
        columns=DISCOVERY_COLUMNS
    )
)

if not issuer_ir_reports_discovered_df.empty:
    issuer_ir_reports_discovered_df = (
        issuer_ir_reports_discovered_df
        .sort_values(
            [
                "asx_code",
                "available_datetime",
                "document_priority",
            ],
            na_position="last",
        )
        .drop_duplicates(
            [
                "asx_code",
                "document_url",
            ]
        )
        .reset_index(drop=True)
    )

if MAX_REPORTS_PER_SECURITY is not None:
    issuer_ir_reports_discovered_df = (
        issuer_ir_reports_discovered_df
        .groupby(
            "asx_code",
            group_keys=False,
        )
        .head(
            int(MAX_REPORTS_PER_SECURITY)
        )
        .reset_index(drop=True)
    )


download_rows = []

for row in tqdm(
    issuer_ir_reports_discovered_df.itertuples(
        index=False
    ),
    total=len(
        issuer_ir_reports_discovered_df
    ),
    desc="Downloading issuer reports",
):
    date_text = (
        pd.Timestamp(
            row.available_datetime
        ).strftime("%Y-%m-%d")
        if pd.notna(
            row.available_datetime
        )
        else "undated"
    )

    safe_title = re.sub(
        r"[^A-Za-z0-9]+",
        "_",
        str(row.headline),
    ).strip("_")[:90]

    filename = (
        f"{row.asx_code}_{date_text}_"
        f"{safe_title}_{row.document_id}.pdf"
    )

    document_path = (
        ASX_DOCUMENT_DIR / filename
    )

    status = "EXISTS"
    error = pd.NA

    if not document_path.exists():
        try:
            response = request_with_retries(
                row.document_url
            )

            content = response.content
            content_type = (
                response.headers.get(
                    "Content-Type",
                    "",
                )
            )

            if not content.startswith(
                b"%PDF"
            ):
                raise RuntimeError(
                    "Report link did not return a PDF. "
                    f"Content-Type={content_type}; "
                    f"final_url={response.url}"
                )

            document_path.write_bytes(
                content
            )

            status = "DOWNLOADED"

            time.sleep(
                IR_LINK_REQUEST_INTERVAL_SECONDS
            )

        except Exception as exc:
            status = "FAILED"
            error = repr(exc)

    download_rows.append({
        "document_id": row.document_id,
        "asx_code": row.asx_code,
        "issuer_name": row.issuer_name,
        "headline": row.headline,
        "document_url": row.document_url,
        "source_page_url": row.source_page_url,
        "available_datetime": row.available_datetime,
        "document_type": row.document_type,
        "document_priority": row.document_priority,
        "document_filename": filename,
        "document_path": str(document_path),
        "document_format": "PDF",
        "document_exists": document_path.is_file(),
        "download_status": status,
        "download_error": error,
        "source_system": row.source_system,
    })


issuer_ir_report_downloads_df = pd.DataFrame(
    download_rows,
    columns=DOWNLOAD_COLUMNS,
)

online_filings_df = (
    issuer_ir_report_downloads_df[
        issuer_ir_report_downloads_df[
            "document_exists"
        ].fillna(False)
    ]
    .copy()
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Official SEC fallback for dual-listed Australian issuers
# ------------------------------------------------------------

SEC_HEADERS = {
    "User-Agent": (
        "Global Automotive Investment Database "
        "research@example.com"
    ),
    "Accept-Encoding": "gzip, deflate",
}

SEC_SESSION = requests.Session()
SEC_SESSION.headers.update(
    SEC_HEADERS
)


def sec_get_json(url):
    response = SEC_SESSION.get(
        url,
        timeout=REQUEST_TIMEOUT_SECONDS,
    )
    response.raise_for_status()
    return response.json()


def sec_get_text(url):
    response = SEC_SESSION.get(
        url,
        timeout=REQUEST_TIMEOUT_SECONDS,
    )
    response.raise_for_status()
    return response.text, response


def classify_sec_filing(
    form,
    filing_text,
):
    form_upper = str(form).upper()
    text_lower = str(
        filing_text
    ).lower()

    if form_upper in {
        "10-K",
        "10-K/A",
        "20-F",
        "20-F/A",
    }:
        return (
            "ANNUAL_REPORT",
            1,
        )

    if form_upper == "10-Q":
        return (
            "INTERIM_REPORT",
            2,
        )

    if form_upper == "6-K":
        if re.search(
            r"(?:half[- ]year|six months ended|"
            r"interim financial|half yearly)",
            text_lower,
        ):
            return (
                "HALF_YEAR_REPORT",
                2,
            )

        if re.search(
            r"(?:annual report|full year|"
            r"year ended)",
            text_lower,
        ):
            return (
                "ANNUAL_REPORT",
                1,
            )

    return (
        "OTHER_SEC_FILING",
        9,
    )


sec_fallback_rows = []
sec_fallback_log_rows = []

for asx_code, cik in (
    SEC_CIK_BY_ASX_CODE.items()
):
    if asx_code not in known_asx_codes:
        continue

    issuer_name = issuer_name_lookup.get(
        asx_code,
        asx_code,
    )

    submissions_url = (
        f"{SEC_SUBMISSIONS_BASE_URL}/"
        f"CIK{cik}.json"
    )

    try:
        submissions = sec_get_json(
            submissions_url
        )

        recent = submissions.get(
            "filings",
            {},
        ).get(
            "recent",
            {},
        )

        recent_df = pd.DataFrame(
            recent
        )

        if recent_df.empty:
            raise RuntimeError(
                "SEC submissions contained no "
                "recent filings."
            )

        permitted_forms = (
            SEC_INCLUDE_FORMS.get(
                asx_code,
                set(),
            )
        )

        recent_df = (
            recent_df[
                recent_df[
                    "form"
                ].isin(
                    permitted_forms
                )
            ]
            .head(
                SEC_MAX_FILINGS_PER_ISSUER
            )
            .copy()
        )

        for filing in recent_df.itertuples(
            index=False
        ):
            accession = str(
                filing.accessionNumber
            )
            accession_no_dashes = (
                accession.replace(
                    "-",
                    "",
                )
            )
            cik_no_leading_zeros = str(
                int(cik)
            )
            primary_document = str(
                filing.primaryDocument
            )

            document_url = (
                f"{SEC_ARCHIVES_BASE_URL}/"
                f"{cik_no_leading_zeros}/"
                f"{accession_no_dashes}/"
                f"{primary_document}"
            )

            try:
                filing_text, response = (
                    sec_get_text(
                        document_url
                    )
                )

                document_type, priority = (
                    classify_sec_filing(
                        filing.form,
                        filing_text,
                    )
                )

                if document_type == (
                    "OTHER_SEC_FILING"
                ):
                    continue

                filing_date = pd.to_datetime(
                    filing.filingDate,
                    errors="coerce",
                    utc=True,
                )

                report_date = pd.to_datetime(
                    filing.reportDate,
                    errors="coerce",
                )

                filename = (
                    f"{asx_code}_"
                    f"{str(filing.filingDate)}_"
                    f"SEC_{str(filing.form).replace('/', '_')}_"
                    f"{accession_no_dashes}.html"
                )

                document_path = (
                    ASX_DOCUMENT_DIR
                    / filename
                )

                document_path.write_text(
                    filing_text,
                    encoding="utf-8",
                    errors="replace",
                )

                document_id = hashlib.sha256(
                    document_url.encode()
                ).hexdigest()[:24]

                sec_fallback_rows.append({
                    "document_id": document_id,
                    "asx_code": asx_code,
                    "issuer_name": issuer_name,
                    "headline": (
                        f"SEC {filing.form} "
                        f"for period "
                        f"{filing.reportDate}"
                    ),
                    "document_url": document_url,
                    "source_page_url": (
                        submissions_url
                    ),
                    "available_datetime": (
                        filing_date
                    ),
                    "document_type": (
                        document_type
                    ),
                    "document_priority": (
                        priority
                    ),
                    "document_filename": (
                        filename
                    ),
                    "document_path": str(
                        document_path
                    ),
                    "document_format": "HTML",
                    "document_exists": True,
                    "download_status": (
                        "DOWNLOADED_SEC"
                    ),
                    "download_error": pd.NA,
                    "source_system": (
                        "SEC_EDGAR"
                    ),
                    "report_period_end": (
                        report_date
                    ),
                    "sec_form": filing.form,
                    "sec_accession_number": (
                        accession
                    ),
                })

                sec_fallback_log_rows.append({
                    "asx_code": asx_code,
                    "form": filing.form,
                    "filing_date": (
                        filing.filingDate
                    ),
                    "report_date": (
                        filing.reportDate
                    ),
                    "document_url": (
                        document_url
                    ),
                    "status": "PASSED",
                    "error": pd.NA,
                })

                time.sleep(
                    IR_LINK_REQUEST_INTERVAL_SECONDS
                )

            except Exception as exc:
                sec_fallback_log_rows.append({
                    "asx_code": asx_code,
                    "form": filing.form,
                    "filing_date": (
                        filing.filingDate
                    ),
                    "report_date": (
                        filing.reportDate
                    ),
                    "document_url": (
                        document_url
                    ),
                    "status": "FAILED",
                    "error": repr(exc),
                })

    except Exception as exc:
        sec_fallback_log_rows.append({
            "asx_code": asx_code,
            "form": pd.NA,
            "filing_date": pd.NA,
            "report_date": pd.NA,
            "document_url": (
                submissions_url
            ),
            "status": "SUBMISSIONS_FAILED",
            "error": repr(exc),
        })


sec_fallback_filings_df = pd.DataFrame(
    sec_fallback_rows
)

sec_fallback_log_df = pd.DataFrame(
    sec_fallback_log_rows
)



local_fallback_rows = []

if ALLOW_LOCAL_PDF_FALLBACK:
    known_paths = set(
        online_filings_df[
            "document_path"
        ].dropna()
    )

    for pdf_path in (
        ASX_DOCUMENT_DIR.rglob("*.pdf")
    ):
        if str(pdf_path) in known_paths:
            continue

        stem_upper = pdf_path.stem.upper()

        matching_codes = [
            code
            for code in known_asx_codes
            if re.search(
                rf"(?:^|[^A-Z0-9])"
                rf"{re.escape(code)}"
                rf"(?:[^A-Z0-9]|$)",
                stem_upper,
            )
        ]

        if len(matching_codes) != 1:
            continue

        asx_code = matching_codes[0]
        document_type, priority = (
            infer_document_type(
                pdf_path.stem,
                str(pdf_path),
            )
        )

        if document_type == (
            "OTHER_FINANCIAL_REPORT"
        ):
            continue

        document_id = hashlib.sha256(
            str(pdf_path.resolve()).encode()
        ).hexdigest()[:24]

        local_fallback_rows.append({
            "document_id": document_id,
            "asx_code": asx_code,
            "issuer_name": issuer_name_lookup.get(
                asx_code,
                asx_code,
            ),
            "headline": pdf_path.stem,
            "document_url": pd.NA,
            "source_page_url": pd.NA,
            "available_datetime": pd.NaT,
            "document_type": document_type,
            "document_priority": priority,
            "document_filename": pdf_path.name,
            "document_path": str(pdf_path),
            "document_format": "PDF",
            "document_exists": True,
            "download_status": "LOCAL_FALLBACK",
            "download_error": pd.NA,
            "source_system": "LOCAL_USER_SUPPLIED_PDF",
        })


local_fallback_filings_df = pd.DataFrame(
    local_fallback_rows,
    columns=DOWNLOAD_COLUMNS,
)

asx_discovered_filings_df = (
    pd.concat(
        [
            online_filings_df,
            sec_fallback_filings_df,
            local_fallback_filings_df,
        ],
        ignore_index=True,
        sort=False,
    )
    .drop_duplicates(
        [
            "asx_code",
            "document_path",
        ]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Deduplicate identical PDFs discovered through multiple pages
# ------------------------------------------------------------

if not asx_discovered_filings_df.empty:
    asx_discovered_filings_df[
        "document_content_sha256"
    ] = asx_discovered_filings_df[
        "document_path"
    ].map(
        lambda value: (
            compute_file_sha256(value)
            if pd.notna(value)
            and Path(value).is_file()
            else pd.NA
        )
    )

    asx_duplicate_pdf_inventory_df = (
        asx_discovered_filings_df[
            asx_discovered_filings_df[
                "document_content_sha256"
            ].notna()
            & asx_discovered_filings_df.duplicated(
                [
                    "asx_code",
                    "document_content_sha256",
                ],
                keep=False,
            )
        ]
        .sort_values(
            [
                "asx_code",
                "document_content_sha256",
                "document_priority",
                "available_datetime",
            ],
            na_position="last",
        )
        .reset_index(drop=True)
    )

    asx_discovered_filings_df = (
        asx_discovered_filings_df
        .sort_values(
            [
                "asx_code",
                "document_priority",
                "available_datetime",
                "document_filename",
            ],
            ascending=[
                True,
                True,
                False,
                True,
            ],
            na_position="last",
        )
        .drop_duplicates(
            [
                "asx_code",
                "document_content_sha256",
            ],
            keep="first",
        )
        .reset_index(drop=True)
    )

else:
    asx_duplicate_pdf_inventory_df = pd.DataFrame(
        columns=DOWNLOAD_COLUMNS
        + ["document_content_sha256"]
    )



issuer_ir_source_coverage_df = (
    pd.DataFrame({
        "asx_code": known_asx_codes,
    })
    .merge(
        issuer_ir_crawl_log_df.groupby(
            "asx_code",
            dropna=False,
        ).agg(
            pages_attempted=("page_url", "count"),
            pages_parsed=(
                "status",
                lambda values: values.eq(
                    "PARSED"
                ).sum(),
            ),
            crawl_failures=(
                "status",
                lambda values: values.eq(
                    "FAILED"
                ).sum(),
            ),
            report_links_found=(
                "report_links_found",
                "sum",
            ),
        ).reset_index(),
        on="asx_code",
        how="left",
    )
    .merge(
        issuer_ir_report_downloads_df.groupby(
            "asx_code",
            dropna=False,
        ).agg(
            download_attempts=(
                "document_id",
                "size",
            ),
            downloaded_or_existing_pdfs=(
                "document_exists",
                "sum",
            ),
            download_failures=(
                "download_status",
                lambda values: values.eq(
                    "FAILED"
                ).sum(),
            ),
        ).reset_index(),
        on="asx_code",
        how="left",
    )
    .fillna(0)
)


issuer_ir_pipeline_quality_df = pd.DataFrame({
    "metric": [
        "asx_codes_in_universe",
        "issuer_ir_pages_attempted",
        "issuer_ir_pages_parsed",
        "issuer_ir_page_failures",
        "report_links_discovered",
        "report_download_attempts",
        "downloaded_or_existing_pdfs",
        "sec_fallback_filings",
        "sec_fallback_issuers",
        "local_fallback_pdfs",
        "duplicate_pdf_rows_identified",
        "total_filings_available",
    ],
    "value": [
        len(known_asx_codes),
        len(issuer_ir_crawl_log_df),
        int(
            issuer_ir_crawl_log_df[
                "status"
            ].eq("PARSED").sum()
        ),
        int(
            issuer_ir_crawl_log_df[
                "status"
            ].eq("FAILED").sum()
        ),
        len(
            issuer_ir_reports_discovered_df
        ),
        len(
            issuer_ir_report_downloads_df
        ),
        int(
            issuer_ir_report_downloads_df[
                "document_exists"
            ].fillna(False).sum()
        ),
        len(
            sec_fallback_filings_df
        ),
        (
            sec_fallback_filings_df[
                "asx_code"
            ].nunique()
            if not sec_fallback_filings_df.empty
            else 0
        ),
        len(
            local_fallback_filings_df
        ),
        len(
            asx_duplicate_pdf_inventory_df
        ),
        len(
            asx_discovered_filings_df
        ),
    ],
})


display(
    issuer_ir_pipeline_quality_df
)

display(
    issuer_ir_source_coverage_df
)

if not sec_fallback_log_df.empty:
    display(
        sec_fallback_log_df
    )

failed_ir_pages_df = (
    issuer_ir_crawl_log_df[
        issuer_ir_crawl_log_df[
            "status"
        ].eq("FAILED")
    ]
    .copy()
)

failed_report_downloads_df = (
    issuer_ir_report_downloads_df[
        issuer_ir_report_downloads_df[
            "download_status"
        ].eq("FAILED")
    ]
    .copy()
)

if not failed_ir_pages_df.empty:
    display(
        failed_ir_pages_df[
            [
                "asx_code",
                "page_url",
                "error",
            ]
        ]
    )

if not failed_report_downloads_df.empty:
    display(
        failed_report_downloads_df[
            [
                "asx_code",
                "headline",
                "document_url",
                "download_error",
            ]
        ]
    )

if asx_discovered_filings_df.empty:
    raise RuntimeError(
        "No annual or half-year report PDFs were obtained from issuer "
        "investor-relations pages, and no valid local fallback PDFs were "
        "found. Review issuer_ir_source_coverage_df, failed_ir_pages_df "
        "and failed_report_downloads_df."
    )

display(
    asx_discovered_filings_df.head(
        100
    )
)

ASX codes passed to issuer IR crawler: ['AKE', 'IGO', 'INR', 'LTM', 'LTR', 'NIC', 'PLS']


Crawling issuer IR websites:   0%|          | 0/7 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.allkem.co'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.allkem.co'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.allkem.co'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: I

/tmp/ipykernel_3057/1587694702.py:1268: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  pd.concat(


,metric,value
0,asx_codes_in_universe,7
1,issuer_ir_pages_attempted,62
2,issuer_ir_pages_parsed,55
3,issuer_ir_page_failures,7
4,report_links_discovered,53
5,report_download_attempts,53
6,downloaded_or_existing_pdfs,43
7,sec_fallback_filings,28
8,sec_fallback_issuers,2
9,local_fallback_pdfs,0


,asx_code,pages_attempted,pages_parsed,crawl_failures,report_links_found,download_attempts,downloaded_or_existing_pdfs,download_failures
0,AKE,2,0,2,0,0.0,0.0,0.0
1,IGO,12,12,0,60,7.0,2.0,5.0
2,INR,18,17,1,0,0.0,0.0,0.0
3,LTM,2,2,0,10,4.0,0.0,4.0
4,LTR,14,14,0,33,4.0,3.0,1.0
5,NIC,10,10,0,89,38.0,38.0,0.0
6,PLS,4,0,4,0,0.0,0.0,0.0


,asx_code,form,filing_date,report_date,document_url,status,error
0,INR,6-K,2026-07-30,2026-07-30,https://www.sec.gov/Archives/edgar/data/189608...,PASSED,<NA>
1,INR,6-K,2026-07-08,2026-07-08,https://www.sec.gov/Archives/edgar/data/189608...,PASSED,<NA>
2,INR,6-K,2026-06-26,2026-06-26,https://www.sec.gov/Archives/edgar/data/189608...,PASSED,<NA>
3,INR,6-K,2026-06-23,2026-06-23,https://www.sec.gov/Archives/edgar/data/189608...,PASSED,<NA>
4,INR,6-K,2026-05-21,2026-05-20,https://www.sec.gov/Archives/edgar/data/189608...,PASSED,<NA>
5,INR,6-K,2026-04-30,2026-04-30,https://www.sec.gov/Archives/edgar/data/189608...,PASSED,<NA>
6,INR,20-F,2026-04-29,2025-12-31,https://www.sec.gov/Archives/edgar/data/189608...,PASSED,<NA>
7,INR,6-K,2026-04-20,2026-04-20,https://www.sec.gov/Archives/edgar/data/189608...,PASSED,<NA>
8,INR,6-K,2026-03-31,2026-03-31,https://www.sec.gov/Archives/edgar/data/189608...,PASSED,<NA>
9,INR,6-K,2026-03-19,2026-03-19,https://www.sec.gov/Archives/edgar/data/189608...,PASSED,<NA>


,asx_code,page_url,error
0,AKE,https://www.allkem.co/investors/reports-and-pr...,"RuntimeError(""Request failed for https://www.a..."
1,AKE,https://www.allkem.co/investors/asx-announceme...,"RuntimeError(""Request failed for https://www.a..."
28,INR,https://www.ioneer.com/investors/reserves-reso...,"RuntimeError(""Request failed for https://www.i..."
58,PLS,https://pilbaraminerals.com.au/investors/repor...,"RuntimeError(""Request failed for https://pilba..."
59,PLS,https://www.pilbaraminerals.com.au/investors/r...,"RuntimeError(""Request failed for https://www.p..."
60,PLS,https://pilbaraminerals.com.au/sustainability/...,"RuntimeError(""Request failed for https://pilba..."
61,PLS,https://pilbaraminerals.com.au/news-stories/fy...,"RuntimeError(""Request failed for https://pilba..."


,asx_code,headline,document_url,download_error
1,IGO,Annual Reports,https://www.igo.com.au/site/investor-center/an...,RuntimeError('Report link did not return a PDF...
3,IGO,Financial Reports,https://www.igo.com.au/site/investor-center/fi...,RuntimeError('Report link did not return a PDF...
4,IGO,Investor Centre,https://www.igo.com.au/site/investor-center,RuntimeError('Report link did not return a PDF...
5,IGO,ASX Announcements,https://www.igo.com.au/site/investor-center/AS...,RuntimeError('Report link did not return a PDF...
6,IGO,Share Registry,https://www.igo.com.au/site/investor-center/sh...,RuntimeError('Report link did not return a PDF...
7,LTM,2026 half year results Announced on Wednesday ...,https://www.riotinto.com/en/invest/financial-n...,RuntimeError('Report link did not return a PDF...
8,LTM,Annual report,https://www.riotinto.com/en/invest/reports/ann...,RuntimeError('Report link did not return a PDF...
9,LTM,Risk management and financial reporting,https://www.riotinto.com/en/about/corporate-go...,RuntimeError('Report link did not return a PDF...
10,LTM,Reports,https://www.riotinto.com/en/invest/reports,RuntimeError('Report link did not return a PDF...
14,LTR,Annual Reporting,https://www.liontown.com/investors/annual-repo...,RuntimeError('Report link did not return a PDF...


,document_id,asx_code,issuer_name,headline,document_url,source_page_url,available_datetime,document_type,document_priority,document_filename,document_path,document_format,document_exists,download_status,download_error,source_system,report_period_end,sec_form,sec_accession_number,document_content_sha256
0,a48f7f0e22d515464e9bdae9,IGO,IGO LIMITED,Annual Report View our 2024 Annual Report,https://www.igo.com.au/site/pdf/f9000c92-0dd9-...,https://www.igo.com.au/site/investor-center/agm,NaT,ANNUAL_REPORT,1,IGO_undated_Annual_Report_View_our_2024_Annual...,/content/drive/MyDrive/Colab Notebooks/00 A1 A...,PDF,True,EXISTS,<NA>,ISSUER_INVESTOR_RELATIONS,NaT,NaN,NaN,60ee0b150f3f1382fe9bf9a257f7b8c1ca91d493ba89e9...
1,ae2e9548f7ef1b235ce5cffe,IGO,IGO LIMITED,19 Feb 2026 December 2025 Half Year Financial ...,https://www.igo.com.au/site/pdf/cea09ae0-556f-...,https://www.igo.com.au/site/investor-center/fi...,2026-02-18 22:00:00+00:00,HALF_YEAR_REPORT,2,IGO_2026-02-18_19_Feb_2026_December_2025_Half_...,/content/drive/MyDrive/Colab Notebooks/00 A1 A...,PDF,True,EXISTS,<NA>,ISSUER_INVESTOR_RELATIONS,NaT,NaN,NaN,dbece8f4731b0debc8490b9acd7568f0717f37ffe37cbf...
2,56efa289896d8de405c5d9da,INR,ioneer Ltd,SEC 6-K for period 2026-07-30,https://www.sec.gov/Archives/edgar/data/189608...,https://data.sec.gov/submissions/CIK0001896084...,2026-07-30 00:00:00+00:00,ANNUAL_REPORT,1,INR_2026-07-30_SEC_6-K_000114036126030118.html,/content/drive/MyDrive/Colab Notebooks/00 A1 A...,HTML,True,DOWNLOADED_SEC,<NA>,SEC_EDGAR,2026-07-30,6-K,0001140361-26-030118,074be30f237626cb3541d7a15a829e3206314198de4a98...
3,af10c0e0705ca2aded54b254,INR,ioneer Ltd,SEC 6-K for period 2026-07-08,https://www.sec.gov/Archives/edgar/data/189608...,https://data.sec.gov/submissions/CIK0001896084...,2026-07-08 00:00:00+00:00,ANNUAL_REPORT,1,INR_2026-07-08_SEC_6-K_000114036126027857.html,/content/drive/MyDrive/Colab Notebooks/00 A1 A...,HTML,True,DOWNLOADED_SEC,<NA>,SEC_EDGAR,2026-07-08,6-K,0001140361-26-027857,7edb00b1055367de24800d91923e106315d13c226dfc97...
4,d989575fb9cce6596209ad97,INR,ioneer Ltd,SEC 6-K for period 2026-06-26,https://www.sec.gov/Archives/edgar/data/189608...,https://data.sec.gov/submissions/CIK0001896084...,2026-06-26 00:00:00+00:00,ANNUAL_REPORT,1,INR_2026-06-26_SEC_6-K_000114036126026511.html,/content/drive/MyDrive/Colab Notebooks/00 A1 A...,HTML,True,DOWNLOADED_SEC,<NA>,SEC_EDGAR,2026-06-26,6-K,0001140361-26-026511,0bcdfb0c9d30c85d7053d625b67f511900061bdb98c4d7...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
65,2bee170240108618cce25476,NIC,Nickel Industries Ltd,Download,https://nickelindustries.com/carbon/assets/202...,https://nickelindustries.com/investor-centre/a...,2019-10-24 22:00:00+00:00,OTHER_FINANCIAL_REPORT,3,NIC_2019-10-24_Download_2bee170240108618cce254...,/content/drive/MyDrive/Colab Notebooks/00 A1 A...,PDF,True,EXISTS,<NA>,ISSUER_INVESTOR_RELATIONS,NaT,NaN,NaN,ffe20d209465172980b49c3acfc0eb1c2054cec60c841b...
66,df2597753ad291cc4de3d9b5,NIC,Nickel Industries Ltd,Download,https://nickelindustries.com/carbon/assets/202...,https://nickelindustries.com/investor-centre/c...,2019-10-24 22:00:00+00:00,OTHER_FINANCIAL_REPORT,3,NIC_2019-10-24_Download_df2597753ad291cc4de3d9...,/content/drive/MyDrive/Colab Notebooks/00 A1 A...,PDF,True,EXISTS,<NA>,ISSUER_INVESTOR_RELATIONS,NaT,NaN,NaN,0ce3b8d4a907ab4f26dc2a08da595c937487d0f74889e2...
67,8b967ff83c6c6d486c795478,NIC,Nickel Industries Ltd,Download,https://nickelindustries.com/carbon/assets/202...,https://nickelindustries.com/investor-centre/a...,2019-08-28 23:00:00+00:00,OTHER_FINANCIAL_REPORT,3,NIC_2019-08-28_Download_8b967ff83c6c6d486c7954...,/content/drive/MyDrive/Colab Notebooks/00 A1 A...,PDF,True,EXISTS,<NA>,ISSUER_INVESTOR_RELATIONS,NaT,NaN,NaN,d0012a734b99671deabb9b0de2e4c382acb391b754733a...
68,2e2161850feff345f8ff677c,NIC,Nickel Industries Ltd,Download,https://nickelindustries.com/carbon/assets/202...,https://nickelindustries.com/investor-centre/c...,2018-10-25

In [6]:
# 6. OPTIONAL METADATA CSV ENRICHMENT

metadata_columns = [
    "document_filename",
    "asx_code",
    "release_datetime_local",
    "headline",
    "document_type",
    "source_url",
    "report_period_end",
]

if ASX_METADATA_PATH.exists():
    metadata_df = pd.read_csv(
        ASX_METADATA_PATH
    )
else:
    metadata_df = pd.DataFrame(
        columns=metadata_columns
    )

for column in metadata_columns:
    if column not in metadata_df.columns:
        metadata_df[column] = pd.NA

metadata_df = metadata_df[
    metadata_columns
].copy()

metadata_df["document_filename"] = (
    metadata_df["document_filename"]
    .astype("string")
    .str.strip()
)

metadata_df["asx_code"] = (
    metadata_df["asx_code"]
    .astype("string")
    .str.upper()
    .str.strip()
)

metadata_df[
    "release_datetime_local"
] = pd.to_datetime(
    metadata_df[
        "release_datetime_local"
    ],
    errors="coerce",
)

metadata_df[
    "report_period_end"
] = pd.to_datetime(
    metadata_df[
        "report_period_end"
    ],
    errors="coerce",
)

australia_filing_metadata_df = (
    asx_discovered_filings_df.merge(
        metadata_df,
        on="document_filename",
        how="left",
        suffixes=("", "_metadata"),
        validate="1:1",
    )
)


# ------------------------------------------------------------
# Preserve and resolve filing format
# ------------------------------------------------------------

if (
    "document_format_metadata"
    in australia_filing_metadata_df.columns
):
    if (
        "document_format"
        not in australia_filing_metadata_df.columns
    ):
        australia_filing_metadata_df[
            "document_format"
        ] = pd.NA

    australia_filing_metadata_df[
        "document_format"
    ] = (
        australia_filing_metadata_df[
            "document_format_metadata"
        ]
        .combine_first(
            australia_filing_metadata_df[
                "document_format"
            ]
        )
    )

if (
    "document_format"
    not in australia_filing_metadata_df.columns
):
    australia_filing_metadata_df[
        "document_format"
    ] = pd.NA


def infer_document_format_from_path(
    document_path,
    current_format,
):
    if pd.notna(current_format):
        value = str(
            current_format
        ).strip().upper()

        if value in {
            "HTML",
            "HTM",
            "XHTML",
        }:
            return "HTML"

        if value == "PDF":
            return "PDF"

    suffix = Path(
        str(document_path)
    ).suffix.lower()

    if suffix in {
        ".html",
        ".htm",
        ".xhtml",
    }:
        return "HTML"

    if suffix == ".pdf":
        return "PDF"

    return "UNKNOWN"


australia_filing_metadata_df[
    "document_format"
] = [
    infer_document_format_from_path(
        document_path,
        current_format,
    )
    for document_path, current_format
    in zip(
        australia_filing_metadata_df[
            "document_path"
        ],
        australia_filing_metadata_df[
            "document_format"
        ],
    )
]



if (
    "report_period_end_metadata"
    in australia_filing_metadata_df.columns
):
    if (
        "report_period_end"
        not in australia_filing_metadata_df.columns
    ):
        australia_filing_metadata_df[
            "report_period_end"
        ] = pd.NaT

    australia_filing_metadata_df[
        "report_period_end"
    ] = (
        pd.to_datetime(
            australia_filing_metadata_df[
                "report_period_end_metadata"
            ],
            errors="coerce",
        )
        .combine_first(
            pd.to_datetime(
                australia_filing_metadata_df[
                    "report_period_end"
                ],
                errors="coerce",
            )
        )
    )


for column in [
    "asx_code",
    "headline",
    "document_type",
]:
    metadata_column = (
        f"{column}_metadata"
    )

    if metadata_column in (
        australia_filing_metadata_df.columns
    ):
        australia_filing_metadata_df[
            column
        ] = (
            australia_filing_metadata_df[
                metadata_column
            ]
            .combine_first(
                australia_filing_metadata_df[
                    column
                ]
            )
        )

australia_filing_metadata_df[
    "release_datetime_metadata_utc"
] = australia_filing_metadata_df[
    "release_datetime_local"
].map(
    lambda value: (
        pd.NaT
        if pd.isna(value)
        else pd.Timestamp(value).tz_localize(
            "Australia/Sydney",
            ambiguous="NaT",
            nonexistent="shift_forward",
        ).tz_convert("UTC")
        if pd.Timestamp(value).tzinfo is None
        else pd.Timestamp(value).tz_convert(
            "UTC"
        )
    )
)

australia_filing_metadata_df[
    "available_datetime"
] = (
    australia_filing_metadata_df[
        "release_datetime_metadata_utc"
    ]
    .combine_first(
        pd.to_datetime(
            australia_filing_metadata_df[
                "available_datetime"
            ],
            utc=True,
            errors="coerce",
        )
    )
)

australia_filing_metadata_df[
    "available_date"
] = (
    australia_filing_metadata_df[
        "available_datetime"
    ]
    .dt.tz_convert(None)
    .dt.normalize()
)

australia_filing_metadata_df[
    "availability_basis"
] = np.select(
    [
        australia_filing_metadata_df[
            "release_datetime_metadata_utc"
        ].notna(),
        australia_filing_metadata_df[
            "available_datetime"
        ].notna(),
    ],
    [
        "USER_SUPPLIED_RELEASE_DATETIME",
        "ISSUER_IR_PAGE_DATE",
    ],
    default="DATE_UNRESOLVED",
)

australia_filing_metadata_df.drop(
    columns=[
        column
        for column in (
            australia_filing_metadata_df.columns
        )
        if column.endswith("_metadata")
    ],
    inplace=True,
    errors="ignore",
)

print(
    "Issuer-IR filing rows:",
    len(asx_discovered_filings_df),
)
print(
    "Metadata enrichment rows:",
    len(metadata_df),
)

display(
    australia_filing_metadata_df[
        [
            "asx_code",
            "available_datetime",
            "availability_basis",
            "document_type",
            "document_format",
            "headline",
            "document_exists",
        ]
    ].head(100)
)

Issuer-IR filing rows: 70
Metadata enrichment rows: 0


/tmp/ipykernel_3057/610292761.py:249: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  .combine_first(


,asx_code,available_datetime,availability_basis,document_type,document_format,headline,document_exists
0,IGO,NaT,DATE_UNRESOLVED,ANNUAL_REPORT,PDF,Annual Report View our 2024 Annual Report,True
1,IGO,2026-02-18 22:00:00+00:00,ISSUER_IR_PAGE_DATE,HALF_YEAR_REPORT,PDF,19 Feb 2026 December 2025 Half Year Financial ...,True
2,INR,2026-07-30 00:00:00+00:00,ISSUER_IR_PAGE_DATE,ANNUAL_REPORT,HTML,SEC 6-K for period 2026-07-30,True
3,INR,2026-07-08 00:00:00+00:00,ISSUER_IR_PAGE_DATE,ANNUAL_REPORT,HTML,SEC 6-K for period 2026-07-08,True
4,INR,2026-06-26 00:00:00+00:00,ISSUER_IR_PAGE_DATE,ANNUAL_REPORT,HTML,SEC 6-K for period 2026-06-26,True
...,...,...,...,...,...,...,...
65,NIC,2019-10-24 22:00:00+00:00,ISSUER_IR_PAGE_DATE,OTHER_FINANCIAL_REPORT,PDF,Download,True
66,NIC,2019-10-24 22:00:00+00:00,ISSUER_IR_PAGE_DATE,OTHER_FINANCIAL_REPORT,PDF,Download,True
67,NIC,2019-08-28 23:00:00+00:00,ISSUER_IR_PAGE_DATE,OTHER_FINANCIAL_REPORT,PDF,Download,True
68,NIC,2018-10-25 22:00:00+00:00,ISSUER_IR_PAGE_DATE,OTHER_FINANCIAL_REPORT,PDF,Download,True


In [7]:
# 7. LINK FILINGS TO THE BLOCK 2 SECURITY MASTER

security_bridge_df = (
    australia_security_universe_df[
        [
            "asx_code",
            "security_id",
            "issuer_id",
            "issuer_name",
            "ticker",
        ]
    ]
    .dropna(subset=["asx_code"])
    .drop_duplicates()
)

australia_filing_metadata_df = (
    australia_filing_metadata_df.merge(
        security_bridge_df,
        on="asx_code",
        how="left",
        validate="m:m",
    )
)

australia_matched_filings_df = (
    australia_filing_metadata_df[
        australia_filing_metadata_df[
            "security_id"
        ].notna()
    ]
    .drop_duplicates(
        [
            "security_id",
            "document_id",
        ]
    )
    .reset_index(drop=True)
)

australia_unmatched_filings_df = (
    australia_filing_metadata_df[
        australia_filing_metadata_df[
            "security_id"
        ].isna()
    ]
    .copy()
    .reset_index(drop=True)
)

australia_filing_link_quality_df = pd.DataFrame({
    "metric": [
        "discovered_pdf_rows",
        "matched_filing_rows",
        "unmatched_filing_rows",
        "linked_unique_securities",
        "linked_unique_issuers",
    ],
    "value": [
        len(australia_filing_metadata_df),
        len(australia_matched_filings_df),
        len(australia_unmatched_filings_df),
        australia_matched_filings_df[
            "security_id"
        ].nunique(),
        australia_matched_filings_df[
            "issuer_id"
        ].nunique(),
    ],
})

display(australia_filing_link_quality_df)

if not australia_unmatched_filings_df.empty:
    display(
        australia_unmatched_filings_df[
            [
                "document_filename",
                "asx_code",
                "asx_code_inference_method",
            ]
        ]
    )

if australia_matched_filings_df.empty:
    raise RuntimeError(
        "No ASX filing could be linked to the Block 2 Security Master. "
        "Rename each PDF so its filename contains a valid ASX code."
    )

australia_filing_metadata_df = (
    australia_matched_filings_df.copy()
)

,metric,value
0,discovered_pdf_rows,70
1,matched_filing_rows,70
2,unmatched_filing_rows,0
3,linked_unique_securities,5
4,linked_unique_issuers,5


In [8]:
# 8. EXTRACT TEXT AND STRUCTURED TABLES FROM LOCAL FILINGS

DATE_HEADER_PATTERN = re.compile(
    r"(?:\b20\d{2}\b|"
    r"\b(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)"
    r"[a-z]*\b|"
    r"\bthree months ended\b|"
    r"\bsix months ended\b|"
    r"\bnine months ended\b|"
    r"\byear ended\b|"
    r"\bas at\b)",
    flags=re.IGNORECASE,
)

UNIT_PATTERN = re.compile(
    r"(?:in\s+thousands|in\s+millions|"
    r"\$['’]?000|US\$['’]?000|A\$['’]?000|"
    r"\$m|US\$m|A\$m|percent|%)",
    flags=re.IGNORECASE,
)


def normalise_html_cell_text(value):
    return re.sub(
        r"\s+",
        " ",
        str(value),
    ).strip()


def infer_scale_from_context(value):
    text = str(value)

    if re.search(
        r"(?:in\s+millions|\$m|US\$m|A\$m)",
        text,
        flags=re.IGNORECASE,
    ):
        return 1_000_000.0

    if re.search(
        r"(?:in\s+thousands|\$['’]?000|"
        r"US\$['’]?000|A\$['’]?000)",
        text,
        flags=re.IGNORECASE,
    ):
        return 1_000.0

    if re.search(
        r"(?:percent|%)",
        text,
        flags=re.IGNORECASE,
    ):
        return 0.01

    return 1.0


def infer_period_type_from_context(value):
    text = str(value).lower()

    if re.search(
        r"(?:three months ended|quarter ended|q[1-4])",
        text,
    ):
        return "QUARTERLY"

    if re.search(
        r"(?:six months ended|half[- ]year|interim)",
        text,
    ):
        return "INTERIM"

    if re.search(
        r"(?:nine months ended)",
        text,
    ):
        return "NINE_MONTH"

    if re.search(
        r"(?:year ended|annual|fiscal year)",
        text,
    ):
        return "ANNUAL"

    if re.search(
        r"(?:as at|balance sheet date)",
        text,
    ):
        return "INSTANT"

    return pd.NA


def infer_period_end_from_context(value):
    text = str(value)

    candidates = pd.to_datetime(
        re.findall(
            r"(?:\d{1,2}\s+"
            r"(?:Jan(?:uary)?|Feb(?:ruary)?|Mar(?:ch)?|Apr(?:il)?|"
            r"May|Jun(?:e)?|Jul(?:y)?|Aug(?:ust)?|Sep(?:tember)?|"
            r"Oct(?:ober)?|Nov(?:ember)?|Dec(?:ember)?)\s+20\d{2}|"
            r"20\d{2}-\d{1,2}-\d{1,2}|"
            r"\d{1,2}/\d{1,2}/20\d{2})",
            text,
            flags=re.IGNORECASE,
        ),
        errors="coerce",
        dayfirst=True,
    )

    candidates = [
        pd.Timestamp(value)
        for value in candidates
        if pd.notna(value)
    ]

    return max(candidates) if candidates else pd.NaT


def extract_structured_html_tables(
    document_id,
    html_text,
):
    """
    Extract structured numeric facts from SEC HTML tables with
    row labels, hierarchical row context, column headers, period
    hints and unit scaling.
    """

    soup = BeautifulSoup(
        html_text,
        "lxml",
    )

    structured_rows = []

    for table_index, table in enumerate(
        soup.find_all("table")
    ):
        rows = table.find_all("tr")

        if not rows:
            continue

        table_context = normalise_html_cell_text(
            " ".join(
                item.get_text(
                    " ",
                    strip=True,
                )
                for item in [
                    table.find_previous(
                        ["h1", "h2", "h3", "h4", "p"]
                    ),
                    table,
                ]
                if item is not None
            )
        )[:3000]

        header_rows = []
        inherited_parent_label = pd.NA

        for row_index, row in enumerate(rows):
            cells = row.find_all(
                ["th", "td"]
            )

            if not cells:
                continue

            cell_texts = [
                normalise_html_cell_text(
                    cell.get_text(
                        " ",
                        strip=True,
                    )
                )
                for cell in cells
            ]

            non_empty = [
                value
                for value in cell_texts
                if value
            ]

            if not non_empty:
                continue

            numeric_flags = [
                bool(
                    re.fullmatch(
                        r"\(?-?\$?\s*"
                        r"\d[\d,]*(?:\.\d+)?"
                        r"\)?(?:\s*%)?",
                        value,
                    )
                )
                for value in cell_texts
            ]

            numeric_count = sum(
                numeric_flags
            )

            if (
                numeric_count == 0
                and any(
                    DATE_HEADER_PATTERN.search(
                        value
                    )
                    for value in cell_texts
                )
            ):
                header_rows.append(
                    cell_texts
                )
                continue

            label = cell_texts[0]

            if (
                not label
                or not re.search(
                    r"[A-Za-z]{3,}",
                    label,
                )
            ):
                continue

            if numeric_count == 0:
                inherited_parent_label = label
                continue

            header_by_column = {}

            for header_row in header_rows[-3:]:
                for column_index, value in enumerate(
                    header_row
                ):
                    if value:
                        header_by_column.setdefault(
                            column_index,
                            [],
                        ).append(value)

            row_context = " | ".join(
                value
                for value in [
                    table_context,
                    inherited_parent_label,
                    " | ".join(
                        cell_value
                        for header_row in header_rows[-3:]
                        for cell_value in header_row
                        if cell_value
                    ) if header_rows else "",
                ]
                if pd.notna(value)
                and str(value).strip()
            )

            for column_index, cell_value in enumerate(
                cell_texts[1:],
                start=1,
            ):
                if not re.fullmatch(
                    r"\(?-?\$?\s*"
                    r"\d[\d,]*(?:\.\d+)?"
                    r"\)?(?:\s*%)?",
                    cell_value,
                ):
                    continue

                column_header = " | ".join(
                    header_by_column.get(
                        column_index,
                        [],
                    )
                )

                full_context = " | ".join(
                    value
                    for value in [
                        row_context,
                        column_header,
                    ]
                    if value
                )

                structured_rows.append({
                    "document_id": document_id,
                    "table_index": table_index,
                    "row_index": row_index,
                    "column_index": column_index,
                    "parent_account_label": (
                        inherited_parent_label
                    ),
                    "account_label": label,
                    "column_header": (
                        column_header
                        if column_header
                        else pd.NA
                    ),
                    "reported_value_raw": cell_value,
                    "source_table_row": " | ".join(
                        cell_texts
                    ),
                    "table_context": table_context,
                    "period_type_inferred": (
                        infer_period_type_from_context(
                            full_context
                        )
                    ),
                    "period_end_inferred": (
                        infer_period_end_from_context(
                            full_context
                        )
                    ),
                    "unit_scale_inferred": (
                        infer_scale_from_context(
                            full_context
                        )
                    ),
                    "extraction_method": (
                        "SEC_HTML_TABLE"
                    ),
                })

    return structured_rows




def resolve_document_format(
    document_path,
    document_format,
):
    if pd.notna(document_format):
        value = str(
            document_format
        ).strip().upper()

        if value in {
            "HTML",
            "HTM",
            "XHTML",
        }:
            return "HTML"

        if value == "PDF":
            return "PDF"

    suffix = Path(
        str(document_path)
    ).suffix.lower()

    if suffix in {
        ".html",
        ".htm",
        ".xhtml",
    }:
        return "HTML"

    if suffix == ".pdf":
        return "PDF"

    return "UNKNOWN"


def extract_filing_text(
    document_id,
    document_path,
    document_format,
):
    path = Path(document_path)

    cache_path = (
        ASX_TEXT_CACHE_DIR
        / f"{document_id}.txt"
    )

    format_upper = resolve_document_format(
        document_path,
        document_format,
    )

    structured_rows = []

    if cache_path.exists():
        text = cache_path.read_text(
            encoding="utf-8",
            errors="replace",
        )

        if format_upper == "HTML":
            html_text = path.read_text(
                encoding="utf-8",
                errors="replace",
            )

            structured_rows = (
                extract_structured_html_tables(
                    document_id,
                    html_text,
                )
            )

        return text, structured_rows, {
            "document_id": document_id,
            "status": "CACHE_HIT",
            "document_path": str(path),
            "document_format": format_upper,
            "text_cache_path": str(
                cache_path
            ),
            "page_count": pd.NA,
            "pages_extracted": pd.NA,
            "failed_page_count": pd.NA,
            "character_count": len(text),
            "structured_table_rows": len(
                structured_rows
            ),
            "error": pd.NA,
        }

    if not path.is_file():
        raise FileNotFoundError(path)

    if format_upper == "HTML":
        html_text = path.read_text(
            encoding="utf-8",
            errors="replace",
        )

        structured_rows = (
            extract_structured_html_tables(
                document_id,
                html_text,
            )
        )

        soup = BeautifulSoup(
            html_text,
            "lxml",
        )

        for element in soup(
            [
                "script",
                "style",
                "noscript",
            ]
        ):
            element.decompose()

        text = "\n".join(
            line.strip()
            for line in soup.get_text(
                "\n"
            ).splitlines()
            if line.strip()
        )

        page_count = pd.NA
        pages_extracted = pd.NA
        failed_pages = pd.NA

    else:
        reader = PdfReader(
            str(path)
        )

        page_count = len(
            reader.pages
        )

        page_limit = page_count

        if MAX_PDF_PAGES is not None:
            page_limit = min(
                page_count,
                int(MAX_PDF_PAGES),
            )

        pages = []
        failed_pages = 0

        for page_number in range(
            page_limit
        ):
            try:
                pages.append(
                    reader.pages[
                        page_number
                    ].extract_text()
                    or ""
                )
            except Exception:
                pages.append("")
                failed_pages += 1

        text = "\n\n".join(
            pages
        )

        pages_extracted = (
            page_limit
        )

    cache_path.write_text(
        text,
        encoding="utf-8",
        errors="replace",
    )

    return text, structured_rows, {
        "document_id": document_id,
        "status": "EXTRACTED",
        "document_path": str(path),
        "document_format": format_upper,
        "text_cache_path": str(
            cache_path
        ),
        "page_count": page_count,
        "pages_extracted": (
            pages_extracted
        ),
        "failed_page_count": (
            failed_pages
        ),
        "character_count": len(text),
        "structured_table_rows": len(
            structured_rows
        ),
        "error": pd.NA,
    }


documents_to_parse_df = (
    australia_filing_metadata_df
    .sort_values(
        [
            "document_priority",
            "available_datetime",
            "document_id",
        ],
        na_position="last",
    )
    .drop_duplicates(
        "document_id"
    )
    .reset_index(drop=True)
)

if MAX_DOCUMENTS_TO_PARSE is not None:
    documents_to_parse_df = (
        documents_to_parse_df
        .head(
            int(
                MAX_DOCUMENTS_TO_PARSE
            )
        )
        .copy()
    )


text_rows = []
structured_table_rows = []
text_logs = []

for row in tqdm(
    documents_to_parse_df.itertuples(
        index=False
    ),
    total=len(
        documents_to_parse_df
    ),
    desc="Extracting filing text and tables",
):
    try:
        resolved_document_format = (
            resolve_document_format(
                row.document_path,
                getattr(
                    row,
                    "document_format",
                    pd.NA,
                ),
            )
        )

        (
            text,
            structured_rows,
            log,
        ) = extract_filing_text(
            row.document_id,
            row.document_path,
            resolved_document_format,
        )

        text_rows.append({
            "document_id": row.document_id,
            "document_content_sha256": getattr(
                row,
                "document_content_sha256",
                pd.NA,
            ),
            "document_text": text,
            "character_count": len(text),
        })

        structured_table_rows.extend(
            structured_rows
        )

        text_logs.append(log)

    except Exception as exc:
        text_logs.append({
            "document_id": row.document_id,
            "status": "FAILED",
            "document_path": row.document_path,
            "document_format": getattr(
                row,
                "document_format",
                pd.NA,
            ),
            "text_cache_path": pd.NA,
            "page_count": pd.NA,
            "pages_extracted": pd.NA,
            "failed_page_count": pd.NA,
            "character_count": 0,
            "structured_table_rows": 0,
            "error": repr(exc),
        })


asx_document_text_df = pd.DataFrame(
    text_rows,
    columns=[
        "document_id",
        "document_content_sha256",
        "document_text",
        "character_count",
    ],
)

sec_html_structured_table_rows_df = (
    pd.DataFrame(
        structured_table_rows,
        columns=[
            "document_id",
            "table_index",
            "row_index",
            "column_index",
            "parent_account_label",
            "account_label",
            "column_header",
            "reported_value_raw",
            "source_table_row",
            "table_context",
            "period_type_inferred",
            "period_end_inferred",
            "unit_scale_inferred",
            "extraction_method",
        ],
    )
)

asx_document_extraction_log_df = (
    pd.DataFrame(
        text_logs
    )
)

asx_document_extraction_quality_df = pd.DataFrame({
    "metric": [
        "documents_requested",
        "documents_with_text",
        "documents_failed",
        "documents_with_empty_text",
        "structured_sec_table_rows",
        "documents_with_structured_sec_rows",
    ],
    "value": [
        len(
            documents_to_parse_df
        ),
        len(
            asx_document_text_df
        ),
        int(
            asx_document_extraction_log_df[
                "status"
            ].eq("FAILED").sum()
        )
        if not asx_document_extraction_log_df.empty
        else 0,
        int(
            asx_document_text_df[
                "character_count"
            ].fillna(0).eq(0).sum()
        )
        if not asx_document_text_df.empty
        else 0,
        len(
            sec_html_structured_table_rows_df
        ),
        (
            sec_html_structured_table_rows_df[
                "document_id"
            ].nunique()
            if not sec_html_structured_table_rows_df.empty
            else 0
        ),
    ],
})

display(
    asx_document_extraction_quality_df
)


sec_html_filing_inventory_df = (
    documents_to_parse_df[
        documents_to_parse_df[
            "document_path"
        ]
        .astype("string")
        .str.lower()
        .str.endswith(
            (
                ".html",
                ".htm",
                ".xhtml",
            )
        )
        | documents_to_parse_df[
            "document_format"
        ]
        .astype("string")
        .str.upper()
        .isin(
            {
                "HTML",
                "HTM",
                "XHTML",
            }
        )
    ]
    .copy()
    .reset_index(drop=True)
)

sec_html_extraction_validation_df = pd.DataFrame({
    "metric": [
        "sec_html_filings_expected",
        "sec_html_filings_with_text",
        "sec_html_structured_rows",
        "sec_html_documents_with_structured_rows",
    ],
    "value": [
        len(
            sec_html_filing_inventory_df
        ),
        int(
            asx_document_extraction_log_df[
                "document_format"
            ]
            .astype("string")
            .str.upper()
            .eq("HTML")
            .sum()
        )
        if not asx_document_extraction_log_df.empty
        else 0,
        len(
            sec_html_structured_table_rows_df
        ),
        (
            sec_html_structured_table_rows_df[
                "document_id"
            ].nunique()
            if not sec_html_structured_table_rows_df.empty
            else 0
        ),
    ],
})

display(
    sec_html_extraction_validation_df
)


sec_html_document_parser_diagnostic_df = (
    asx_document_extraction_log_df[
        asx_document_extraction_log_df[
            "document_format"
        ]
        .astype("string")
        .str.upper()
        .eq("HTML")
    ]
    .copy()
    .merge(
        sec_html_structured_table_rows_df
        .groupby(
            "document_id",
            dropna=False,
        )
        .size()
        .rename(
            "structured_row_count"
        )
        .reset_index(),
        on="document_id",
        how="left",
    )
)

sec_html_document_parser_diagnostic_df[
    "structured_row_count"
] = (
    sec_html_document_parser_diagnostic_df[
        "structured_row_count"
    ]
    .fillna(0)
    .astype("int64")
)

display(
    sec_html_document_parser_diagnostic_df[
        [
            "document_id",
            "status",
            "document_path",
            "character_count",
            "structured_row_count",
            "error",
        ]
    ]
)


if (
    len(
        sec_html_filing_inventory_df
    ) > 0
    and len(
        sec_html_structured_table_rows_df
    ) == 0
):
    raise RuntimeError(
        "SEC HTML filings were present, but the structured "
        "HTML-table parser produced zero rows. Review "
        "sec_html_filing_inventory_df and "
        "asx_document_extraction_log_df."
    )


if asx_document_text_df.empty:
    raise RuntimeError(
        "No filing text was extracted. Review "
        "asx_document_extraction_log_df."
    )

Extracting filing text and tables:   0%|          | 0/70 [00:00<?, ?it/s]

/tmp/ipykernel_3057/638659439.py:132: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(
/tmp/ipykernel_3057/638659439.py:98: UserWarning: Parsing dates in %m/%d/%Y format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  candidates = pd.to_datetime(


,metric,value
0,documents_requested,70
1,documents_with_text,70
2,documents_failed,0
3,documents_with_empty_text,0
4,structured_sec_table_rows,10931
5,documents_with_structured_sec_rows,10


,metric,value
0,sec_html_filings_expected,28
1,sec_html_filings_with_text,28
2,sec_html_structured_rows,10931
3,sec_html_documents_with_structured_rows,10


,document_id,status,document_path,character_count,structured_row_count,error
0,ffedde888888c9740c883104,CACHE_HIT,/content/drive/MyDrive/Colab Notebooks/00 A1 A...,646681,1181,<NA>
1,70a206f95dabd63a0aa879a9,CACHE_HIT,/content/drive/MyDrive/Colab Notebooks/00 A1 A...,8190,1,<NA>
2,5f131a6f23ebc2ecf04a86d8,CACHE_HIT,/content/drive/MyDrive/Colab Notebooks/00 A1 A...,171949,379,<NA>
3,b92414e765b304306a3f8963,CACHE_HIT,/content/drive/MyDrive/Colab Notebooks/00 A1 A...,735673,1225,<NA>
4,36c134ebb9fccc0bea4dd262,CACHE_HIT,/content/drive/MyDrive/Colab Notebooks/00 A1 A...,155172,458,<NA>
5,16445dccdb8bdffb49c53dd1,CACHE_HIT,/content/drive/MyDrive/Colab Notebooks/00 A1 A...,1042,0,<NA>
6,8df9cd7227acb2626224d383,CACHE_HIT,/content/drive/MyDrive/Colab Notebooks/00 A1 A...,1041,0,<NA>
7,97dc2be64a74aa8df773fa7e,CACHE_HIT,/content/drive/MyDrive/Colab Notebooks/00 A1 A...,486592,2757,<NA>
8,ee39c1c09312ba8f80a0dfc8,CACHE_HIT,/content/drive/MyDrive/Colab Notebooks/00 A1 A...,1036,0,<NA>
9,1cfee4659be9ce8bfb50689f,CACHE_HIT,/content/drive/MyDrive/Colab Notebooks/00 A1 A...,1401,0,<NA>


In [9]:
# 9. AUSTRALIAN GENERIC ACCOUNT-LABEL DICTIONARY

AUSTRALIA_ACCOUNT_MAPPINGS = [
    # Income statement
    ("revenue", r"^(?:total )?(?:revenue|sales revenue|sales|operating revenue|revenue from contracts with customers|sales and other operating revenue)$", 1),
    ("cost_of_revenue", r"^(?:cost of sales|cost of goods sold|cost of revenue|cost of operations)$", 1),
    ("gross_profit", r"^gross profit$", 1),
    ("other_income", r"^(?:other income|other operating income)$", 1),
    ("selling_general_admin", r"^(?:selling general and administrative expenses|selling, general and administrative expenses|administrative expenses|general and administrative expenses)$", 1),
    ("research_development", r"^(?:research and development expense|research and development expenses)$", 1),
    ("depreciation_amortisation", r"^(?:depreciation and amortisation|depreciation and amortization|depreciation expense|amortisation expense|amortization expense)$", 1),
    ("operating_income", r"^(?:operating profit|operating income|profit from operations|income from operations|earnings before interest and tax|ebit)$", 1),
    ("finance_income", r"^(?:finance income|interest income)$", 1),
    ("finance_costs", r"^(?:finance costs|finance expense|interest expense|net finance costs)$", 1),
    ("profit_before_tax", r"^(?:profit|loss|income) before (?:income )?tax$", 1),
    ("income_tax_expense", r"^(?:income tax expense|tax expense|income tax benefit|benefit from income taxes)$", 1),
    ("net_income", r"^(?:profit|loss|net income) for the (?:year|period)|net profit|net income|net loss$", 1),
    ("net_income_attributable_parent", r"^(?:profit|loss|net income) attributable to (?:owners|shareholders|members) of the parent$", 1),
    ("ebitda", r"^(?:ebitda|earnings before interest,? tax,? depreciation and amortisation|earnings before interest,? taxes,? depreciation and amortization)$", 1),

    # Balance sheet assets
    ("cash_and_cash_equivalents", r"^(?:cash and cash equivalents|cash at bank and on hand|cash and deposits|cash)$", 1),
    ("restricted_cash", r"^(?:restricted cash|restricted cash and cash equivalents)$", 1),
    ("trade_receivables", r"^(?:trade receivables|trade and other receivables|accounts receivable|receivables)$", 1),
    ("inventory", r"^(?:inventories|inventory)$", 1),
    ("prepaid_expenses", r"^(?:prepayments|prepaid expenses|prepaid expenses and other current assets)$", 1),
    ("other_current_assets", r"^other current assets$", 1),
    ("property_plant_equipment", r"^(?:property,? plant and equipment|plant and equipment|property and equipment)$", 1),
    ("right_of_use_assets", r"^(?:right-of-use assets|right of use assets)$", 1),
    ("intangible_assets", r"^(?:intangible assets|intangibles)$", 1),
    ("goodwill", r"^goodwill$", 1),
    ("investments", r"^(?:investments|financial assets|other investments)$", 1),
    ("deferred_tax_assets", r"^deferred tax assets?$", 1),
    ("total_current_assets", r"^total current assets$", 1),
    ("total_non_current_assets", r"^total non-current assets$", 1),
    ("total_assets", r"^total assets$", 1),

    # Balance sheet liabilities
    ("trade_payables", r"^(?:trade payables|trade and other payables|accounts payable|accounts payable and accrued liabilities)$", 1),
    ("accrued_expenses", r"^(?:accrued expenses|accrued liabilities|accruals)$", 1),
    ("short_term_debt", r"^(?:current borrowings|short-term borrowings|short term debt|current debt|current interest-bearing liabilities)$", 1),
    ("long_term_debt", r"^(?:non-current borrowings|long-term borrowings|long term debt|non-current debt|non-current interest-bearing liabilities)$", 1),
    ("lease_liabilities_current", r"^(?:current lease liabilities|lease liabilities current)$", 1),
    ("lease_liabilities_non_current", r"^(?:non-current lease liabilities|lease liabilities non-current)$", 1),
    ("provisions_current", r"^(?:current provisions|provisions current)$", 1),
    ("provisions_non_current", r"^(?:non-current provisions|provisions non-current)$", 1),
    ("deferred_tax_liabilities", r"^deferred tax liabilities?$", 1),
    ("other_current_liabilities", r"^other current liabilities$", 1),
    ("other_non_current_liabilities", r"^other non-current liabilities$", 1),
    ("total_current_liabilities", r"^total current liabilities$", 1),
    ("total_non_current_liabilities", r"^total non-current liabilities$", 1),
    ("total_liabilities", r"^total liabilities$", 1),

    # Equity
    ("share_capital", r"^(?:share capital|issued capital|common stock|ordinary shares)$", 1),
    ("additional_paid_in_capital", r"^(?:additional paid-in capital|additional paid in capital|share premium)$", 1),
    ("retained_earnings", r"^(?:retained earnings|accumulated losses|accumulated deficit)$", 1),
    ("reserves", r"^(?:reserves|other reserves)$", 1),
    ("treasury_stock", r"^(?:treasury stock|treasury shares)$", 1),
    ("non_controlling_interests", r"^(?:non-controlling interests|noncontrolling interests)$", 1),
    ("total_equity", r"^(?:total equity|shareholders'? equity|stockholders'? equity|equity attributable to owners)$", 1),

    # Cash flow
    ("operating_cash_flow", r"^(?:net cash (?:from|provided by|generated from) operating activities|cash flows? from operating activities|net cash provided by operating activities)$", 1),
    ("investing_cash_flow", r"^(?:net cash (?:used in|from|provided by) investing activities|cash flows? from investing activities)$", 1),
    ("financing_cash_flow", r"^(?:net cash (?:from|used in|provided by) financing activities|cash flows? from financing activities)$", 1),
    ("capital_expenditure", r"^(?:(?:payments for|purchase of|purchases of) property,? plant and equipment|capital expenditure|capital expenditures)$", 1),
    ("acquisitions", r"^(?:acquisitions of businesses|business acquisitions|payments for acquisitions)$", 1),
    ("dividends_paid", r"^(?:dividends paid|payments of dividends)$", 1),
    ("share_repurchases", r"^(?:share repurchases|repurchase of shares|purchase of treasury shares)$", 1),
    ("debt_issued", r"^(?:proceeds from borrowings|proceeds from debt|borrowings received)$", 1),
    ("debt_repaid", r"^(?:repayment of borrowings|repayments of debt|debt repayments)$", 1),
    ("cash_change", r"^(?:net increase|decrease in cash and cash equivalents|net change in cash and cash equivalents)$", 1),

    # Per-share and distributions
    ("basic_eps", r"^(?:basic earnings per share|basic eps|basic loss per share)$", 1),
    ("diluted_eps", r"^(?:diluted earnings per share|diluted eps|diluted loss per share)$", 1),
    ("dividends_per_share", r"^(?:dividends per share|dividend per share)$", 1),

    # Generic operating / analytical items
    ("working_capital", r"^working capital$", 1),
    ("net_debt", r"^net debt$", 1),
    ("total_debt", r"^(?:total debt|total borrowings)$", 1),
    ("current_ratio", r"^current ratio$", 1),
    ("book_value_per_share", r"^(?:book value per share|net tangible assets per share)$", 1),
]

australia_source_account_mapping_df = pd.DataFrame(
    AUSTRALIA_ACCOUNT_MAPPINGS,
    columns=[
        "standard_concept",
        "account_label_regex",
        "mapping_priority",
    ],
)

if not shared_concept_dictionary_df.empty:
    canonical_columns = [
        column
        for column in shared_concept_dictionary_df.columns
        if column not in {
            "account_label_regex",
            "mapping_priority",
        }
    ]

    australia_standard_concept_dictionary_df = (
        shared_concept_dictionary_df[
            canonical_columns
        ]
        .drop_duplicates(
            "standard_concept"
        )
        .merge(
            australia_source_account_mapping_df,
            on="standard_concept",
            how="right",
            validate="1:1",
        )
    )

else:
    australia_standard_concept_dictionary_df = (
        australia_source_account_mapping_df.copy()
    )

print(
    "Australian generic mapped concepts:",
    australia_standard_concept_dictionary_df[
        "standard_concept"
    ].nunique(),
)

# ------------------------------------------------------------
# Prepare an unambiguous Block 9 synonym lookup for Australia
# ------------------------------------------------------------

def normalise_block9_synonym_label(value):
    if pd.isna(value):
        return pd.NA

    text = str(value).casefold()
    text = re.sub(r"[\u00a0\s]+", " ", text)
    text = re.sub(
        r"[^\w\u3400-\u9fff]+",
        " ",
        text,
        flags=re.UNICODE,
    )
    text = re.sub(r"\s+", " ", text).strip()

    return text if text else pd.NA


if block9_accepted_synonym_registry_raw_df.empty:
    block9_accepted_synonym_registry_df = pd.DataFrame(
        columns=block9_accepted_synonym_registry_raw_df.columns
    )
    block9_synonym_registry_conflicts_df = pd.DataFrame(
        columns=block9_accepted_synonym_registry_raw_df.columns
    )

else:
    registry_work = (
        block9_accepted_synonym_registry_raw_df.copy()
    )

    registry_work[
        "normalised_source_account_label"
    ] = registry_work[
        "normalised_source_account_label"
    ].map(normalise_block9_synonym_label)

    registry_work[
        "standard_concept"
    ] = (
        registry_work["standard_concept"]
        .astype("string")
        .str.strip()
    )

    registry_work[
        "accepted_observations"
    ] = pd.to_numeric(
        registry_work["accepted_observations"],
        errors="coerce",
    ).fillna(0)

    valid_australia_concepts = set(
        australia_standard_concept_dictionary_df[
            "standard_concept"
        ]
        .dropna()
        .astype("string")
    )

    registry_work = registry_work.loc[
        registry_work[
            "normalised_source_account_label"
        ].notna()
        & registry_work[
            "standard_concept"
        ].isin(valid_australia_concepts)
    ].copy()

    concept_counts = (
        registry_work.groupby(
            "normalised_source_account_label",
            dropna=False,
        )["standard_concept"]
        .nunique(dropna=True)
    )

    unambiguous_labels = set(
        concept_counts.loc[concept_counts == 1].index
    )
    conflicting_labels = set(
        concept_counts.loc[concept_counts > 1].index
    )

    block9_synonym_registry_conflicts_df = (
        registry_work.loc[
            registry_work[
                "normalised_source_account_label"
            ].isin(conflicting_labels)
        ]
        .sort_values(
            [
                "normalised_source_account_label",
                "accepted_observations",
            ],
            ascending=[True, False],
        )
        .reset_index(drop=True)
    )

    block9_accepted_synonym_registry_df = (
        registry_work.loc[
            registry_work[
                "normalised_source_account_label"
            ].isin(unambiguous_labels)
        ]
        .sort_values(
            [
                "normalised_source_account_label",
                "accepted_observations",
                "last_accepted_at_utc",
            ],
            ascending=[True, False, False],
            na_position="last",
        )
        .drop_duplicates(
            "normalised_source_account_label",
            keep="first",
        )
        .reset_index(drop=True)
    )


block9_synonym_lookup = (
    block9_accepted_synonym_registry_df
    .set_index("normalised_source_account_label")
    .to_dict(orient="index")
    if not block9_accepted_synonym_registry_df.empty
    else {}
)

block9_synonym_registry_status_df = pd.DataFrame([
    {
        **block9_synonym_registry_load_status,
        "usable_registry_rows": int(
            len(block9_accepted_synonym_registry_df)
        ),
        "usable_unique_labels": int(
            block9_accepted_synonym_registry_df[
                "normalised_source_account_label"
            ].nunique(dropna=True)
            if not block9_accepted_synonym_registry_df.empty
            else 0
        ),
        "conflicting_registry_rows": int(
            len(block9_synonym_registry_conflicts_df)
        ),
        "conflicting_unique_labels": int(
            block9_synonym_registry_conflicts_df[
                "normalised_source_account_label"
            ].nunique(dropna=True)
            if not block9_synonym_registry_conflicts_df.empty
            else 0
        ),
    }
])

display(block9_synonym_registry_status_df)

Australian generic mapped concepts: 69


,status,manifest_path,loaded_table_name,loaded_rows,error,usable_registry_rows,usable_unique_labels,conflicting_registry_rows,conflicting_unique_labels
0,LOADED,/content/drive/MyDrive/Colab Notebooks/00 A1 A...,accepted_synonym_registry,531,<NA>,239,239,2,1


In [10]:
# 10. EXTRACT ACCOUNT / VALUE CANDIDATES

NUMBER_PATTERN = re.compile(
    r"(?P<value>\(?-?\$?\s*"
    r"\d[\d,]*(?:\.\d+)?\)?"
    r"(?:\s*%)?)"
)

LINE_PATTERN = re.compile(
    r"^\s*(?P<label>[A-Za-z][A-Za-z0-9&/(),.'’\-\s]{2,120}?)"
    r"\s{2,}"
    r"(?P<value>\(?-?\$?\s*"
    r"\d[\d,]*(?:\.\d+)?\)?"
    r"(?:\s*%)?)"
)

ACCOUNT_STOPWORDS = {
    "note",
    "notes",
    "total",
    "subtotal",
    "current",
    "non current",
    "non-current",
    "continued",
    "restated",
    "audited",
    "unaudited",
}


def normalise_account_label(value):
    text = re.sub(
        r"\s+",
        " ",
        str(value),
    ).strip()

    text = re.sub(
        r"^[\d.\-]+\s*",
        "",
        text,
    )

    return text.strip(
        " .:-"
    )


def label_is_eligible(value):
    text = normalise_account_label(
        value
    )

    if not text:
        return False

    lower = text.lower()

    if lower in ACCOUNT_STOPWORDS:
        return False

    if len(text) < 4 or len(text) > 120:
        return False

    if len(text.split()) > 12:
        return False

    if re.search(
        r"(?:www\.|https?://|@)",
        text,
        flags=re.IGNORECASE,
    ):
        return False

    if not re.search(
        r"[A-Za-z]{3,}",
        text,
    ):
        return False

    return True


def parse_number(value):
    text = str(value).strip()

    is_percent = text.endswith(
        "%"
    )

    text = text.rstrip(
        "%"
    ).strip()

    negative = (
        text.startswith("(")
        and text.endswith(")")
    )

    text = (
        text.replace("(", "")
        .replace(")", "")
        .replace(",", "")
        .replace("$", "")
        .strip()
    )

    number = pd.to_numeric(
        text,
        errors="coerce",
    )

    if pd.isna(number):
        return np.nan

    numeric_value = (
        -float(number)
        if negative
        else float(number)
    )

    if is_percent:
        return (
            numeric_value
            / 100.0
        )

    return numeric_value


candidate_rows = []

# ------------------------------------------------------------
# Flattened filing-text candidates
# ------------------------------------------------------------

for document in tqdm(
    asx_document_text_df.itertuples(
        index=False
    ),
    total=len(
        asx_document_text_df
    ),
    desc="Extracting text-line account candidates",
):
    for line_number, raw_line in enumerate(
        str(
            document.document_text
        ).splitlines()
    ):
        if not raw_line.strip():
            continue

        if len(raw_line) > 260:
            continue

        match = LINE_PATTERN.search(
            raw_line
        )

        if match:
            label = normalise_account_label(
                match.group(
                    "label"
                )
            )
            value_raw = match.group(
                "value"
            )

        else:
            line = raw_line.strip()

            numbers = list(
                NUMBER_PATTERN.finditer(
                    line
                )
            )

            if not numbers:
                continue

            last = numbers[-1]

            label = normalise_account_label(
                line[
                    :last.start()
                ]
            )
            value_raw = last.group(
                "value"
            )

        if not label_is_eligible(
            label
        ):
            continue

        value = parse_number(
            value_raw
        )

        if pd.isna(value):
            continue

        candidate_rows.append({
            "document_id": (
                document.document_id
            ),
            "parent_account_label": pd.NA,
            "account_label": label,
            "reported_value_raw": (
                value_raw
            ),
            "reported_value": value,
            "unit_scale": 1.0,
            "scaled_reported_value": value,
            "column_header": pd.NA,
            "period_type_inferred": pd.NA,
            "period_end_inferred": pd.NaT,
            "source_line": re.sub(
                r"\s+",
                " ",
                raw_line,
            ).strip(),
            "source_line_number": (
                line_number
            ),
            "extraction_method": (
                "TEXT_LINE"
            ),
        })


# ------------------------------------------------------------
# Structured SEC HTML-table candidates
# ------------------------------------------------------------

if not sec_html_structured_table_rows_df.empty:
    for row in tqdm(
        sec_html_structured_table_rows_df.itertuples(
            index=False
        ),
        total=len(
            sec_html_structured_table_rows_df
        ),
        desc="Extracting SEC table account candidates",
    ):
        label = normalise_account_label(
            row.account_label
        )

        if not label_is_eligible(
            label
        ):
            continue

        value = parse_number(
            row.reported_value_raw
        )

        if pd.isna(value):
            continue

        unit_scale = (
            pd.to_numeric(
                row.unit_scale_inferred,
                errors="coerce",
            )
            if pd.notna(
                row.unit_scale_inferred
            )
            else 1.0
        )

        if pd.isna(unit_scale):
            unit_scale = 1.0

        candidate_rows.append({
            "document_id": (
                row.document_id
            ),
            "parent_account_label": (
                row.parent_account_label
            ),
            "account_label": label,
            "reported_value_raw": (
                row.reported_value_raw
            ),
            "reported_value": value,
            "unit_scale": (
                float(unit_scale)
            ),
            "scaled_reported_value": (
                value
                * float(unit_scale)
            ),
            "column_header": (
                row.column_header
            ),
            "period_type_inferred": (
                row.period_type_inferred
            ),
            "period_end_inferred": (
                row.period_end_inferred
            ),
            "source_line": (
                row.source_table_row
            ),
            "source_line_number": (
                row.row_index
            ),
            "extraction_method": (
                row.extraction_method
            ),
        })


australia_account_candidates_raw_df = (
    pd.DataFrame(
        candidate_rows,
        columns=[
            "document_id",
            "parent_account_label",
            "account_label",
            "reported_value_raw",
            "reported_value",
            "unit_scale",
            "scaled_reported_value",
            "column_header",
            "period_type_inferred",
            "period_end_inferred",
            "source_line",
            "source_line_number",
            "extraction_method",
        ],
    )
)


australia_account_candidate_quality_df = pd.DataFrame({
    "metric": [
        "total_candidate_rows",
        "text_line_candidate_rows",
        "sec_html_table_candidate_rows",
        "documents_with_candidates",
        "unique_account_labels",
        "candidates_with_period_end",
        "candidates_with_non_unit_scale",
    ],
    "value": [
        len(
            australia_account_candidates_raw_df
        ),
        int(
            australia_account_candidates_raw_df[
                "extraction_method"
            ].eq(
                "TEXT_LINE"
            ).sum()
        ),
        int(
            australia_account_candidates_raw_df[
                "extraction_method"
            ].eq(
                "SEC_HTML_TABLE"
            ).sum()
        ),
        (
            australia_account_candidates_raw_df[
                "document_id"
            ].nunique()
            if not australia_account_candidates_raw_df.empty
            else 0
        ),
        (
            australia_account_candidates_raw_df[
                "account_label"
            ].nunique()
            if not australia_account_candidates_raw_df.empty
            else 0
        ),
        int(
            pd.to_datetime(
                australia_account_candidates_raw_df[
                    "period_end_inferred"
                ],
                errors="coerce",
            ).notna().sum()
        ),
        int(
            australia_account_candidates_raw_df[
                "unit_scale"
            ].fillna(1.0).ne(1.0).sum()
        ),
    ],
})

display(
    australia_account_candidate_quality_df
)

Extracting text-line account candidates:   0%|          | 0/70 [00:00<?, ?it/s]

Extracting SEC table account candidates:   0%|          | 0/10931 [00:00<?, ?it/s]

,metric,value
0,total_candidate_rows,46744
1,text_line_candidate_rows,36261
2,sec_html_table_candidate_rows,10483
3,documents_with_candidates,70
4,unique_account_labels,15432
5,candidates_with_period_end,825
6,candidates_with_non_unit_scale,8480


In [11]:
# 11. MAP ACCOUNT LABELS AND BUILD STANDARDISED FACTS

required_mapping_columns = {
    "standard_concept",
    "account_label_regex",
    "mapping_priority",
}

missing_mapping_columns = (
    required_mapping_columns.difference(
        australia_standard_concept_dictionary_df.columns
    )
)

if missing_mapping_columns:
    raise RuntimeError(
        "Australian mapping dictionary is missing columns: "
        f"{sorted(missing_mapping_columns)}"
    )


# ------------------------------------------------------------
# Compile the existing deterministic Australian mappings
# ------------------------------------------------------------

compiled_mapping_rows = []

for row in (
    australia_standard_concept_dictionary_df[
        [
            "standard_concept",
            "account_label_regex",
            "mapping_priority",
        ]
    ]
    .dropna(
        subset=[
            "standard_concept",
            "account_label_regex",
        ]
    )
    .drop_duplicates()
    .itertuples(index=False)
):
    try:
        pattern = re.compile(
            str(row.account_label_regex),
            flags=re.IGNORECASE,
        )

        compiled_mapping_rows.append({
            "standard_concept": row.standard_concept,
            "mapping_priority": (
                pd.to_numeric(
                    row.mapping_priority,
                    errors="coerce",
                )
                if pd.notna(row.mapping_priority)
                else 1
            ),
            "account_label_regex": row.account_label_regex,
            "compiled_pattern": pattern,
        })

    except re.error as exc:
        print(
            "Skipping invalid regex:",
            row.account_label_regex,
            repr(exc),
        )


def deterministic_mapping_candidates(label):
    text = str(label).strip()
    matches = []

    for row in compiled_mapping_rows:
        if row["compiled_pattern"].fullmatch(text):
            matches.append({
                "standard_concept": row[
                    "standard_concept"
                ],
                "mapping_priority": row[
                    "mapping_priority"
                ],
                "account_label_regex": row[
                    "account_label_regex"
                ],
                "mapping_method": "FULL_REGEX_MATCH",
                "mapping_score": 99.0,
                "mapping_is_accepted": True,
                "mapping_source": "BLOCK_8_RULE_ENGINE",
                "block9_registry_accepted_observations": pd.NA,
                "block9_registry_unique_issuers": pd.NA,
                "block9_registry_unique_filings": pd.NA,
                "block9_registry_generation": pd.NA,
            })

    return matches


def block9_registry_mapping_candidate(label):
    """Return one exact, unambiguous mapping learned in Block 9."""
    normalised_label = normalise_block9_synonym_label(label)

    if pd.isna(normalised_label):
        return None

    registry_row = block9_synonym_lookup.get(
        normalised_label
    )

    if not registry_row:
        return None

    standard_concept = registry_row.get(
        "standard_concept"
    )

    if pd.isna(standard_concept):
        return None

    return {
        "standard_concept": standard_concept,
        "mapping_priority": 2,
        "account_label_regex": pd.NA,
        "mapping_method": "BLOCK9_ACCEPTED_SYNONYM",
        "mapping_score": 100.0,
        "mapping_is_accepted": True,
        "mapping_source": "BLOCK_9_ACCEPTED_REGISTRY",
        "block9_registry_accepted_observations": (
            registry_row.get("accepted_observations")
        ),
        "block9_registry_unique_issuers": (
            registry_row.get("unique_issuers")
        ),
        "block9_registry_unique_filings": (
            registry_row.get("unique_filings")
        ),
        "block9_registry_generation": (
            registry_row.get("registry_generation")
        ),
    }


# ------------------------------------------------------------
# Map labels in conservative precedence order:
# deterministic regex -> exact Block 9 synonym -> unresolved
# ------------------------------------------------------------

unique_labels = (
    australia_account_candidates_raw_df[
        "account_label"
    ]
    .dropna()
    .astype("string")
    .drop_duplicates()
    .tolist()
)

mapping_rows = []

for label in tqdm(
    unique_labels,
    desc="Mapping Australian account labels",
):
    deterministic_matches = (
        deterministic_mapping_candidates(label)
    )

    if deterministic_matches:
        for match in deterministic_matches:
            mapping_rows.append({
                "account_label": label,
                **match,
            })
        continue

    registry_match = (
        block9_registry_mapping_candidate(label)
    )

    if registry_match is not None:
        mapping_rows.append({
            "account_label": label,
            **registry_match,
        })


observed_mapping_columns = [
    "account_label",
    "standard_concept",
    "mapping_priority",
    "account_label_regex",
    "mapping_method",
    "mapping_score",
    "mapping_is_accepted",
    "mapping_source",
    "block9_registry_accepted_observations",
    "block9_registry_unique_issuers",
    "block9_registry_unique_filings",
    "block9_registry_generation",
]

australia_observed_account_mapping_df = pd.DataFrame(
    mapping_rows,
    columns=observed_mapping_columns,
)


australia_block9_registry_matches_df = (
    australia_observed_account_mapping_df.loc[
        australia_observed_account_mapping_df[
            "mapping_method"
        ]
        .astype("string")
        .eq("BLOCK9_ACCEPTED_SYNONYM")
    ]
    .copy()
    .reset_index(drop=True)
    if not australia_observed_account_mapping_df.empty
    else pd.DataFrame(columns=observed_mapping_columns)
)


if australia_observed_account_mapping_df.empty:
    australia_fundamentals_mapped_df = pd.DataFrame()
    australia_fundamentals_standardised_df = pd.DataFrame()
    australia_mapping_alternatives_df = pd.DataFrame()

else:
    mapped_df = (
        australia_account_candidates_raw_df.merge(
            australia_observed_account_mapping_df,
            on="account_label",
            how="inner",
            validate="m:m",
        )
    )

    filing_bridge_columns = [
        "document_id",
        "security_id",
        "issuer_id",
        "issuer_name",
        "asx_code",
        "headline",
        "document_type",
        "report_period_end",
        "available_datetime",
        "available_date",
        "availability_basis",
        "source_system",
        "document_content_sha256",
    ]

    filing_bridge_df = (
        australia_filing_metadata_df[
            [
                column
                for column in filing_bridge_columns
                if column
                in australia_filing_metadata_df.columns
            ]
        ]
        .drop_duplicates("document_id")
        .copy()
    )

    mapped_df = mapped_df.merge(
        filing_bridge_df,
        on="document_id",
        how="left",
        validate="m:1",
    )

    if "extraction_method" not in mapped_df.columns:
        mapped_df["extraction_method"] = "TEXT_LINE"

    mapped_df[
        "_extraction_priority"
    ] = (
        mapped_df["extraction_method"]
        .map({
            "SEC_HTML_TABLE": 0,
            "TEXT_LINE": 1,
        })
        .fillna(2)
        .astype("int8")
    )

    mapped_df[
        "source_quality_score"
    ] = np.select(
        [
            mapped_df["source_system"].eq("SEC_EDGAR")
            & mapped_df["extraction_method"].eq(
                "SEC_HTML_TABLE"
            ),
            mapped_df["source_system"].eq(
                "ISSUER_INVESTOR_RELATIONS"
            )
            & mapped_df["extraction_method"].eq(
                "TEXT_LINE"
            ),
            mapped_df["source_system"].eq(
                "LOCAL_USER_SUPPLIED_PDF"
            ),
        ],
        [1.00, 0.90, 0.80],
        default=0.70,
    )

    mapped_df[
        "resolved_period_end"
    ] = (
        pd.to_datetime(
            mapped_df["period_end_inferred"],
            errors="coerce",
        )
        .combine_first(
            pd.to_datetime(
                mapped_df["report_period_end"],
                errors="coerce",
            )
        )
    )

    mapped_df[
        "resolved_period_type"
    ] = (
        mapped_df["period_type_inferred"]
        .astype("string")
        .replace({
            "<NA>": pd.NA,
            "nan": pd.NA,
        })
        .combine_first(
            pd.Series(
                np.where(
                    mapped_df["document_type"]
                    .astype("string")
                    .isin({
                        "HALF_YEAR_REPORT",
                        "APPENDIX_4D",
                        "INTERIM_REPORT",
                    }),
                    "INTERIM",
                    "ANNUAL",
                ),
                index=mapped_df.index,
                dtype="string",
            )
        )
    )

    mapped_df["period_type"] = (
        mapped_df["resolved_period_type"]
    )
    mapped_df["source_filing_id"] = (
        mapped_df["document_id"]
    )

    mapped_df[
        "canonical_value"
    ] = (
        pd.to_numeric(
            mapped_df["scaled_reported_value"],
            errors="coerce",
        )
        .combine_first(
            pd.to_numeric(
                mapped_df["reported_value"],
                errors="coerce",
            )
        )
    )

    mapped_df["filing_date"] = (
        mapped_df["available_datetime"]
    )

    # Explicit production provenance consumed by Block 10.
    mapped_df["source_region"] = "AUSTRALIA"
    mapped_df[
        "source_table"
    ] = "australia_fundamentals_standardised_df"
    mapped_df["source_row_number"] = pd.to_numeric(
        mapped_df["source_line_number"],
        errors="coerce",
    )

    selection_key = [
        "issuer_id",
        "security_id",
        "document_id",
        "standard_concept",
    ]

    mapped_df = mapped_df.sort_values(
        selection_key
        + [
            "source_quality_score",
            "_extraction_priority",
            "mapping_priority",
            "source_line_number",
        ],
        ascending=[
            True,
            True,
            True,
            True,
            False,
            True,
            True,
            True,
        ],
        na_position="last",
    )

    mapped_df[
        "is_selected_standard_fact"
    ] = ~mapped_df.duplicated(
        selection_key,
        keep="first",
    )

    australia_fundamentals_mapped_df = (
        mapped_df.reset_index(drop=True)
    )

    australia_fundamentals_standardised_df = (
        mapped_df.loc[
            mapped_df["is_selected_standard_fact"]
        ]
        .copy()
        .reset_index(drop=True)
    )

    australia_mapping_alternatives_df = (
        mapped_df.loc[
            ~mapped_df["is_selected_standard_fact"]
        ]
        .copy()
        .reset_index(drop=True)
    )


mapped_labels = set(
    australia_observed_account_mapping_df[
        "account_label"
    ]
    .dropna()
    .astype("string")
)

unmapped_df = (
    australia_account_candidates_raw_df.loc[
        ~australia_account_candidates_raw_df[
            "account_label"
        ]
        .astype("string")
        .isin(mapped_labels)
    ]
    .copy()
)

australia_unmapped_account_inventory_df = (
    unmapped_df
    .groupby(
        "account_label",
        dropna=False,
    )
    .agg(
        candidate_rows=("reported_value", "size"),
        document_count=("document_id", "nunique"),
        example_line=("source_line", "first"),
    )
    .reset_index()
    .sort_values(
        ["document_count", "candidate_rows"],
        ascending=[False, False],
    )
    .reset_index(drop=True)
)


registry_production_rows = 0
if (
    not australia_fundamentals_standardised_df.empty
    and "mapping_method"
    in australia_fundamentals_standardised_df.columns
):
    registry_production_rows = int(
        australia_fundamentals_standardised_df[
            "mapping_method"
        ]
        .astype("string")
        .eq("BLOCK9_ACCEPTED_SYNONYM")
        .sum()
    )


australia_mapping_quality_df = pd.DataFrame({
    "metric": [
        "raw_account_candidate_rows",
        "unique_account_labels",
        "mapped_unique_account_labels",
        "mapped_fact_rows",
        "selected_standardised_fact_rows",
        "mapping_alternative_rows",
        "observed_standard_concepts",
        "unmapped_unique_account_labels",
        "block9_registry_candidate_labels",
        "block9_registry_rows_in_standardised_fundamentals",
    ],
    "value": [
        len(australia_account_candidates_raw_df),
        australia_account_candidates_raw_df[
            "account_label"
        ].nunique(),
        australia_observed_account_mapping_df[
            "account_label"
        ].nunique(),
        len(australia_fundamentals_mapped_df),
        len(australia_fundamentals_standardised_df),
        len(australia_mapping_alternatives_df),
        (
            australia_fundamentals_standardised_df[
                "standard_concept"
            ].nunique()
            if not australia_fundamentals_standardised_df.empty
            else 0
        ),
        len(australia_unmapped_account_inventory_df),
        len(australia_block9_registry_matches_df),
        registry_production_rows,
    ],
})

display(australia_mapping_quality_df)

Mapping Australian account labels:   0%|          | 0/15432 [00:00<?, ?it/s]

,metric,value
0,raw_account_candidate_rows,46744
1,unique_account_labels,15432
2,mapped_unique_account_labels,112
3,mapped_fact_rows,1636
4,selected_standardised_fact_rows,659
5,mapping_alternative_rows,977
6,observed_standard_concepts,47
7,unmapped_unique_account_labels,15320
8,block9_registry_candidate_labels,35
9,block9_registry_rows_in_standardised_fundamentals,78


In [12]:
# 12. POINT-IN-TIME HELPERS AND COVERAGE QA

def get_australia_fundamentals_as_of(
    as_of_date,
    issuer_ids=None,
    security_ids=None,
    standard_concepts=None,
):
    frame = (
        australia_fundamentals_standardised_df
        .copy()
    )

    if frame.empty:
        return frame

    cutoff = pd.Timestamp(
        as_of_date
    )

    cutoff = (
        cutoff.tz_localize("UTC")
        if cutoff.tzinfo is None
        else cutoff.tz_convert("UTC")
    )

    frame = frame[
        pd.to_datetime(
            frame[
                "available_datetime"
            ],
            utc=True,
            errors="coerce",
        )
        <= cutoff
    ]

    if issuer_ids is not None:
        frame = frame[
            frame[
                "issuer_id"
            ].isin(
                set(
                    issuer_ids
                )
            )
        ]

    if security_ids is not None:
        frame = frame[
            frame[
                "security_id"
            ].isin(
                set(
                    security_ids
                )
            )
        ]

    if standard_concepts is not None:
        frame = frame[
            frame[
                "standard_concept"
            ].isin(
                set(
                    standard_concepts
                )
            )
        ]

    return frame.reset_index(
        drop=True
    )


standardised_fact_schema_df = pd.DataFrame({
    "column_name": list(
        australia_fundamentals_standardised_df.columns
    ),
    "dtype": [
        str(dtype)
        for dtype in (
            australia_fundamentals_standardised_df.dtypes
        )
    ],
})

required_coverage_columns = {
    "issuer_id",
    "standard_concept",
    "document_id",
    "available_datetime",
}

missing_coverage_columns = (
    required_coverage_columns.difference(
        australia_fundamentals_standardised_df.columns
    )
)

if missing_coverage_columns:
    raise RuntimeError(
        "Standardised fundamentals are missing required "
        "coverage columns: "
        f"{sorted(missing_coverage_columns)}"
    )


CORE_COVERAGE_CONCEPTS = [
    "revenue",
    "ebitda",
    "operating_income",
    "net_income",
    "cash_and_cash_equivalents",
    "total_assets",
    "total_liabilities",
    "total_equity",
    "operating_cash_flow",
    "capital_expenditure",
    "total_debt",
    "basic_eps",
]

issuer_concept_presence_df = (
    australia_fundamentals_standardised_df[
        [
            "issuer_id",
            "standard_concept",
        ]
    ]
    .dropna(
        subset=[
            "issuer_id",
            "standard_concept",
        ]
    )
    .drop_duplicates()
    .assign(
        available=1
    )
)

if issuer_concept_presence_df.empty:
    australia_fundamental_coverage_df = (
        australia_issuer_universe_df[
            [
                "issuer_id",
                "issuer_name",
            ]
        ]
        .copy()
    )

else:
    coverage_pivot_df = (
        issuer_concept_presence_df[
            issuer_concept_presence_df[
                "standard_concept"
            ].isin(
                CORE_COVERAGE_CONCEPTS
            )
        ]
        .pivot_table(
            index=[
                "issuer_id",
            ],
            columns=(
                "standard_concept"
            ),
            values="available",
            aggfunc="max",
            fill_value=0,
        )
        .reset_index()
    )

    coverage_pivot_df.columns.name = None

    australia_fundamental_coverage_df = (
        australia_issuer_universe_df[
            [
                "issuer_id",
                "issuer_name",
            ]
        ]
        .merge(
            coverage_pivot_df,
            on=[
                "issuer_id",
            ],
            how="left",
            validate="1:1",
        )
    )

for concept in CORE_COVERAGE_CONCEPTS:
    if (
        concept
        not in australia_fundamental_coverage_df.columns
    ):
        australia_fundamental_coverage_df[
            concept
        ] = 0

australia_fundamental_coverage_df[
    CORE_COVERAGE_CONCEPTS
] = (
    australia_fundamental_coverage_df[
        CORE_COVERAGE_CONCEPTS
    ]
    .fillna(0)
    .astype("int8")
)

australia_fundamental_coverage_df[
    "core_concepts_available"
] = (
    australia_fundamental_coverage_df[
        CORE_COVERAGE_CONCEPTS
    ].sum(axis=1)
)

australia_fundamental_coverage_df[
    "core_coverage_ratio"
] = (
    australia_fundamental_coverage_df[
        "core_concepts_available"
    ]
    / len(
        CORE_COVERAGE_CONCEPTS
    )
)

issuer_fact_summary_df = (
    australia_fundamentals_standardised_df
    .dropna(
        subset=[
            "issuer_id",
        ]
    )
    .groupby(
        [
            "issuer_id",
        ],
        dropna=False,
    )
    .agg(
        standardised_fact_rows=(
            "standard_concept",
            "size",
        ),
        standard_concepts=(
            "standard_concept",
            "nunique",
        ),
        filing_documents=(
            "document_id",
            "nunique",
        ),
        earliest_available_datetime=(
            "available_datetime",
            "min",
        ),
        latest_available_datetime=(
            "available_datetime",
            "max",
        ),
        mean_source_quality=(
            "source_quality_score",
            "mean",
        ),
    )
    .reset_index()
)

australia_issuer_fundamental_coverage_report_df = (
    australia_issuer_universe_df[
        [
            "issuer_id",
            "issuer_name",
            "security_count",
            "asx_codes",
        ]
    ]
    .merge(
        issuer_fact_summary_df,
        on=[
            "issuer_id",
        ],
        how="left",
        validate="1:1",
    )
    .merge(
        australia_fundamental_coverage_df[
            [
                "issuer_id",
                "core_concepts_available",
                "core_coverage_ratio",
            ]
        ],
        on="issuer_id",
        how="left",
    )
)

australia_issuer_fundamental_coverage_report_df[
    "has_standardised_fundamentals"
] = (
    australia_issuer_fundamental_coverage_report_df[
        "standardised_fact_rows"
    ]
    .fillna(0)
    .gt(0)
)



australia_source_system_coverage_df = (
    australia_fundamentals_standardised_df
    .groupby(
        [
            "issuer_id",
            "source_system",
        ],
        dropna=False,
    )
    .agg(
        standardised_fact_rows=(
            "standard_concept",
            "size",
        ),
        standard_concepts=(
            "standard_concept",
            "nunique",
        ),
        filing_documents=(
            "document_id",
            "nunique",
        ),
    )
    .reset_index()
    .merge(
        australia_issuer_universe_df[
            [
                "issuer_id",
                "issuer_name",
                "asx_codes",
            ]
        ],
        on="issuer_id",
        how="left",
        validate="m:1",
    )
)

australia_mapping_quality_df = pd.DataFrame({
    "metric": [
        "raw_account_candidates",
        "unique_account_labels",
        "mapped_account_labels",
        "standardised_fact_rows",
        "observed_standard_concepts",
        "unmapped_account_labels",
        "issuers_with_standardised_fundamentals",
        "mean_core_coverage_ratio",
    ],
    "value": [
        len(
            australia_account_candidates_raw_df
        ),
        australia_account_candidates_raw_df[
            "account_label"
        ].nunique(),
        (
            australia_observed_account_mapping_df[
                "account_label"
            ].nunique()
            if not australia_observed_account_mapping_df.empty
            else 0
        ),
        len(
            australia_fundamentals_standardised_df
        ),
        (
            australia_fundamentals_standardised_df[
                "standard_concept"
            ].nunique()
            if not australia_fundamentals_standardised_df.empty
            else 0
        ),
        len(
            australia_unmapped_account_inventory_df
        ),
        int(
            australia_issuer_fundamental_coverage_report_df[
                "has_standardised_fundamentals"
            ].sum()
        ),
        float(
            australia_fundamental_coverage_df[
                "core_coverage_ratio"
            ].mean()
        ),
    ],
})

australia_filing_coverage_report_df = pd.DataFrame({
    "metric": [
        "australian_securities",
        "australian_issuers",
        "discovered_filings",
        "linked_filing_rows",
        "documents_with_text",
        "standardised_fact_rows",
        "issuers_with_standardised_fundamentals",
    ],
    "value": [
        australia_security_universe_df[
            "security_id"
        ].nunique(),
        australia_issuer_universe_df[
            "issuer_id"
        ].nunique(),
        len(
            asx_discovered_filings_df
        ),
        len(
            australia_filing_metadata_df
        ),
        len(
            asx_document_text_df
        ),
        len(
            australia_fundamentals_standardised_df
        ),
        int(
            australia_issuer_fundamental_coverage_report_df[
                "has_standardised_fundamentals"
            ].sum()
        ),
    ],
})

display(
    australia_mapping_quality_df
)

display(
    australia_filing_coverage_report_df
)

display(
    australia_issuer_fundamental_coverage_report_df
)

display(
    australia_fundamental_coverage_df
)

,metric,value
0,raw_account_candidates,46744.000000
1,unique_account_labels,15432.000000
2,mapped_account_labels,112.000000
3,standardised_fact_rows,659.000000
4,observed_standard_concepts,47.000000
5,unmapped_account_labels,15320.000000
6,issuers_with_standardised_fundamentals,5.000000
7,mean_core_coverage_ratio,0.345238


,metric,value
0,australian_securities,7
1,australian_issuers,7
2,discovered_filings,70
3,linked_filing_rows,70
4,documents_with_text,70
5,standardised_fact_rows,659
6,issuers_with_standardised_fundamentals,5


,issuer_id,issuer_name,security_count,asx_codes,standardised_fact_rows,standard_concepts,filing_documents,earliest_available_datetime,latest_available_datetime,mean_source_quality,core_concepts_available,core_coverage_ratio,has_standardised_fundamentals
0,GAI_2C802D47EF667C123F15,ioneer Ltd,1,INR,52.0,26.0,2.0,2025-10-22 00:00:00+00:00,2026-04-29 00:00:00+00:00,1.0,6,0.500000,True
1,GAI_35C1E52F44016A33F594,ALLKEM LIMITED,1,AKE,NaN,NaN,NaN,NaT,NaT,NaN,0,0.000000,False
2,GAI_B2E4ECD04E4E4CD53273,Nickel Industries Ltd,1,NIC,436.0,37.0,35.0,2018-08-15 23:00:00+00:00,2026-04-23 23:00:00+00:00,0.9,8,0.666667,True
3,GAI_B79EA297FBC86386AD61,LIONTOWN RESOURCES LIMITED,1,LTR,19.0,19.0,1.0,2025-09-24 23:00:00+00:00,2025-09-24 23:00:00+00:00,0.9,6,0.500000,True
4,GAI_C905CFCC658D674645F2,PILBARA MINERALS LIMITED,1,PLS,NaN,NaN,NaN,NaT,NaT,NaN,0,0.000000,False
5,GAI_ECA2EA117981D9E0FF2B,IGO LIMITED,1,IGO,14.0,11.0,2.0,2026-02-18 22:00:00+00:00,2026-02-18 22:00:00+00:00,0.9,3,0.250000,True
6,GAI_FDF7FFF16173ED43E31A,Arcadium Lithium PLC,1,LTM,138.0,34.0,5.0,2024-02-29 00:00:00+00:00,2025-02-27 00:00:00+00:00,1.0,6,0.500000,True


,issuer_id,issuer_name,capital_expenditure,cash_and_cash_equivalents,net_income,operating_cash_flow,revenue,total_assets,total_debt,total_equity,total_liabilities,ebitda,operating_income,basic_eps,core_concepts_available,core_coverage_ratio
0,GAI_2C802D47EF667C123F15,ioneer Ltd,0,1,1,0,0,1,1,1,1,0,0,0,6,0.500000
1,GAI_35C1E52F44016A33F594,ALLKEM LIMITED,0,0,0,0,0,0,0,0,0,0,0,0,0,0.000000
2,GAI_B2E4ECD04E4E4CD53273,Nickel Industries Ltd,1,1,1,1,1,1,0,1,1,0,0,0,8,0.666667
3,GAI_B79EA297FBC86386AD61,LIONTOWN RESOURCES LIMITED,0,1,1,0,1,1,0,1,1,0,0,0,6,0.500000
4,GAI_C905CFCC658D674645F2,PILBARA MINERALS LIMITED,0,0,0,0,0,0,0,0,0,0,0,0,0,0.000000
5,GAI_ECA2EA117981D9E0FF2B,IGO LIMITED,0,1,1,0,1,0,0,0,0,0,0,0,3,0.250000
6,GAI_FDF7FFF16173ED43E31A,Arcadium Lithium PLC,0,1,1,0,1,1,0,1,1,0,0,0,6,0.500000


In [13]:
# 13. OUTPUT CONTRACT

block_8_data = {
    "australia_security_universe_df": australia_security_universe_df,
    "australia_issuer_universe_df": australia_issuer_universe_df,

    "issuer_ir_crawl_log_df": issuer_ir_crawl_log_df,
    "issuer_ir_reports_discovered_df": issuer_ir_reports_discovered_df,
    "issuer_ir_report_downloads_df": issuer_ir_report_downloads_df,
    "sec_fallback_filings_df": sec_fallback_filings_df,
    "sec_fallback_log_df": sec_fallback_log_df,
    "local_fallback_filings_df": local_fallback_filings_df,
    "issuer_ir_source_coverage_df": issuer_ir_source_coverage_df,
    "issuer_ir_pipeline_quality_df": issuer_ir_pipeline_quality_df,
    "asx_discovered_filings_df": asx_discovered_filings_df,
    "asx_duplicate_pdf_inventory_df": asx_duplicate_pdf_inventory_df,
    "australia_filing_metadata_df": australia_filing_metadata_df,
    "australia_matched_filings_df": australia_matched_filings_df,
    "australia_unmatched_filings_df": australia_unmatched_filings_df,
    "australia_filing_link_quality_df": australia_filing_link_quality_df,

    "asx_document_text_df": asx_document_text_df,
    "sec_html_structured_table_rows_df": sec_html_structured_table_rows_df,
    "australia_source_system_coverage_df": australia_source_system_coverage_df,
    "sec_html_extraction_validation_df": sec_html_extraction_validation_df,
    "sec_html_document_parser_diagnostic_df": sec_html_document_parser_diagnostic_df,
    "sec_html_filing_inventory_df": sec_html_filing_inventory_df,
    "australia_account_candidate_quality_df": australia_account_candidate_quality_df,
    "asx_document_extraction_log_df": asx_document_extraction_log_df,
    "asx_document_extraction_quality_df": asx_document_extraction_quality_df,

    "australia_source_account_mapping_df": australia_source_account_mapping_df,
    "australia_standard_concept_dictionary_df": australia_standard_concept_dictionary_df,
    "australia_account_candidates_raw_df": australia_account_candidates_raw_df,
    "australia_observed_account_mapping_df": australia_observed_account_mapping_df,
    "australia_block9_registry_matches_df": australia_block9_registry_matches_df,
    "block9_accepted_synonym_registry_df": block9_accepted_synonym_registry_df,
    "block9_synonym_registry_conflicts_df": block9_synonym_registry_conflicts_df,
    "block9_synonym_registry_status_df": block9_synonym_registry_status_df,

    "australia_fundamentals_mapped_df": australia_fundamentals_mapped_df,
    "australia_mapping_alternatives_df": australia_mapping_alternatives_df,
    "australia_fundamentals_standardised_df": australia_fundamentals_standardised_df,
    "australia_unmapped_account_inventory_df": australia_unmapped_account_inventory_df,

    "australia_mapping_quality_df": australia_mapping_quality_df,
    "australia_filing_coverage_report_df": australia_filing_coverage_report_df,
    "australia_issuer_fundamental_coverage_report_df": australia_issuer_fundamental_coverage_report_df,
    "australia_fundamental_coverage_df": australia_fundamental_coverage_df,
    "standardised_fact_schema_df": standardised_fact_schema_df,
}

for name, dataframe in block_8_data.items():
    print(
        f"{name}: "
        f"{len(dataframe):,} rows × "
        f"{len(dataframe.columns):,} columns"
    )

print("Block 8 transformations complete.")

australia_security_universe_df: 7 rows × 35 columns
australia_issuer_universe_df: 7 rows × 4 columns
issuer_ir_crawl_log_df: 62 rows × 10 columns
issuer_ir_reports_discovered_df: 53 rows × 12 columns
issuer_ir_report_downloads_df: 53 rows × 16 columns
sec_fallback_filings_df: 28 rows × 19 columns
sec_fallback_log_df: 28 rows × 7 columns
local_fallback_filings_df: 0 rows × 16 columns
issuer_ir_source_coverage_df: 7 rows × 8 columns
issuer_ir_pipeline_quality_df: 12 rows × 2 columns
asx_discovered_filings_df: 70 rows × 20 columns
asx_duplicate_pdf_inventory_df: 2 rows × 20 columns
australia_filing_metadata_df: 70 rows × 29 columns
australia_matched_filings_df: 70 rows × 29 columns
australia_unmatched_filings_df: 0 rows × 29 columns
australia_filing_link_quality_df: 5 rows × 2 columns
asx_document_text_df: 70 rows × 4 columns
sec_html_structured_table_rows_df: 10,931 rows × 14 columns
australia_source_system_coverage_df: 5 rows × 7 columns
sec_html_extraction_validation_df: 4 rows × 2 col

In [14]:
# 14. MEMORY-SAFE PERSISTENCE AND VALIDATION

def prepare_for_parquet(dataframe):
    output = dataframe.copy()

    for column in output.columns:
        if output[column].dtype != "object":
            continue

        non_missing = output[column].dropna()

        if non_missing.empty:
            continue

        sample_types = (
            non_missing.head(10_000)
            .map(type)
            .nunique()
        )

        if sample_types > 1:
            output[column] = (
                output[column]
                .astype("string")
            )

    return output


def persist_dataframe(
    table_name,
    dataframe,
):
    path = (
        BLOCK_8_OUTPUT_DIR
        / f"{table_name}.parquet"
    )

    if (
        path.exists()
        and not OVERWRITE_PERSISTED_OUTPUTS
    ):
        raise FileExistsError(path)

    safe_df = prepare_for_parquet(
        dataframe
    )

    safe_df.to_parquet(
        path,
        index=False,
        engine="pyarrow",
        compression="snappy",
    )

    metadata = {
        "table_name": table_name,
        "path": str(path),
        "row_count": int(len(safe_df)),
        "column_count": int(
            len(safe_df.columns)
        ),
        "file_size_bytes": int(
            path.stat().st_size
        ),
    }

    del safe_df
    gc.collect()

    return metadata


if PERSIST_BLOCK_8_OUTPUTS:
    manifest_rows = []

    for table_name, dataframe in (
        block_8_data.items()
    ):
        print(
            "Persisting:",
            table_name,
        )

        manifest_rows.append(
            persist_dataframe(
                table_name,
                dataframe,
            )
        )

    block_8_manifest = {
        "block": 8,
        "block_name": (
            "Australia / ASX fundamentals"
        ),
        "created_at_utc": datetime.now(
            timezone.utc
        ).isoformat(),
        "output_directory": str(
            BLOCK_8_OUTPUT_DIR
        ),
        "ai_assistance_used": False,
        "downstream_ai_qc_block": 9,
        "tables": manifest_rows,
    }

    with BLOCK_8_MANIFEST_PATH.open(
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            block_8_manifest,
            file,
            indent=2,
        )

    validation_rows = []

    for record in manifest_rows:
        path = Path(record["path"])
        parquet_file = pq.ParquetFile(path)

        persisted_rows = (
            parquet_file.metadata.num_rows
        )

        expected_rows = len(
            block_8_data[
                record["table_name"]
            ]
        )

        if persisted_rows != expected_rows:
            raise RuntimeError(
                "Persistence row-count mismatch "
                f"for {record['table_name']}."
            )

        validation_rows.append({
            "table_name": record[
                "table_name"
            ],
            "expected_rows": expected_rows,
            "persisted_rows": persisted_rows,
            "status": "PASSED",
        })

    block_8_validation_report_df = (
        pd.DataFrame(validation_rows)
    )

    display(block_8_validation_report_df)

    print(
        "Block 8 persistence validation passed."
    )



# ------------------------------------------------------------
# Block 9 feedback contract validation
# ------------------------------------------------------------

required_feedback_columns = [
    "issuer_id",
    "security_id",
    "document_id",
    "standard_concept",
    "mapping_method",
    "mapping_score",
    "mapping_is_accepted",
    "mapping_source",
    "source_region",
    "source_table",
    "source_row_number",
]

missing_feedback_columns = [
    column
    for column in required_feedback_columns
    if column
    not in australia_fundamentals_standardised_df.columns
]

if missing_feedback_columns:
    raise RuntimeError(
        "Australia standardised fundamentals are missing Block 9 "
        "feedback/provenance columns: "
        f"{missing_feedback_columns}"
    )

registry_candidate_rows = len(
    australia_block9_registry_matches_df
)

registry_production_rows = int(
    australia_fundamentals_standardised_df[
        "mapping_method"
    ]
    .astype("string")
    .eq("BLOCK9_ACCEPTED_SYNONYM")
    .sum()
)

registry_production_unique_labels = int(
    australia_fundamentals_standardised_df.loc[
        australia_fundamentals_standardised_df[
            "mapping_method"
        ]
        .astype("string")
        .eq("BLOCK9_ACCEPTED_SYNONYM"),
        "account_label",
    ].nunique(dropna=True)
)

registry_production_unique_concepts = int(
    australia_fundamentals_standardised_df.loc[
        australia_fundamentals_standardised_df[
            "mapping_method"
        ]
        .astype("string")
        .eq("BLOCK9_ACCEPTED_SYNONYM"),
        "standard_concept",
    ].nunique(dropna=True)
)

block8_block9_feedback_diagnostic_df = pd.DataFrame({
    "metric": [
        "registry_load_status",
        "registry_rows_loaded",
        "usable_registry_rows",
        "conflicting_registry_rows",
        "australian_registry_candidate_labels",
        "registry_rows_in_standardised_fundamentals",
        "registry_unique_labels_in_standardised_fundamentals",
        "registry_unique_concepts_in_standardised_fundamentals",
        "standardised_fundamental_rows",
        "remaining_unmapped_inventory_rows",
    ],
    "value": [
        block9_synonym_registry_status_df[
            "status"
        ].iloc[0],
        block9_synonym_registry_status_df[
            "loaded_rows"
        ].iloc[0],
        len(block9_accepted_synonym_registry_df),
        len(block9_synonym_registry_conflicts_df),
        registry_candidate_rows,
        registry_production_rows,
        registry_production_unique_labels,
        registry_production_unique_concepts,
        len(australia_fundamentals_standardised_df),
        len(australia_unmapped_account_inventory_df),
    ],
})

display(block8_block9_feedback_diagnostic_df)

if APPLY_BLOCK_9_SYNONYM_REGISTRY:
    registry_status = str(
        block9_synonym_registry_status_df[
            "status"
        ].iloc[0]
    )

    if registry_status != "LOADED":
        raise RuntimeError(
            "Block 9 synonym feedback was required but the registry "
            f"did not load. Status: {registry_status}"
        )

    if len(block9_accepted_synonym_registry_df) == 0:
        raise RuntimeError(
            "Block 9 registry loaded but contained no usable, "
            "unambiguous concepts supported by the Australia schema."
        )

    if registry_candidate_rows == 0:
        print(
            "WARNING: Block 9 loaded successfully, but no currently "
            "observed Australian labels matched the accepted registry."
        )

    elif registry_production_rows == 0:
        raise RuntimeError(
            "Australian labels matched the Block 9 registry, but no "
            "registry-derived rows reached the standardised facts."
        )

    else:
        print(
            "Block 8 Block 9 feedback diagnostic passed."
        )

Persisting: australia_security_universe_df
Persisting: australia_issuer_universe_df
Persisting: issuer_ir_crawl_log_df
Persisting: issuer_ir_reports_discovered_df
Persisting: issuer_ir_report_downloads_df
Persisting: sec_fallback_filings_df
Persisting: sec_fallback_log_df
Persisting: local_fallback_filings_df
Persisting: issuer_ir_source_coverage_df
Persisting: issuer_ir_pipeline_quality_df
Persisting: asx_discovered_filings_df
Persisting: asx_duplicate_pdf_inventory_df
Persisting: australia_filing_metadata_df
Persisting: australia_matched_filings_df
Persisting: australia_unmatched_filings_df
Persisting: australia_filing_link_quality_df
Persisting: asx_document_text_df
Persisting: sec_html_structured_table_rows_df
Persisting: australia_source_system_coverage_df
Persisting: sec_html_extraction_validation_df
Persisting: sec_html_document_parser_diagnostic_df
Persisting: sec_html_filing_inventory_df
Persisting: australia_account_candidate_quality_df
Persisting: asx_document_extraction_log

,table_name,expected_rows,persisted_rows,status
0,australia_security_universe_df,7,7,PASSED
1,australia_issuer_universe_df,7,7,PASSED
2,issuer_ir_crawl_log_df,62,62,PASSED
3,issuer_ir_reports_discovered_df,53,53,PASSED
4,issuer_ir_report_downloads_df,53,53,PASSED
5,sec_fallback_filings_df,28,28,PASSED
6,sec_fallback_log_df,28,28,PASSED
7,local_fallback_filings_df,0,0,PASSED
8,issuer_ir_source_coverage_df,7,7,PASSED
9,issuer_ir_pipeline_quality_df,12,12,PASSED


Block 8 persistence validation passed.


,metric,value
0,registry_load_status,LOADED
1,registry_rows_loaded,531
2,usable_registry_rows,239
3,conflicting_registry_rows,2
4,australian_registry_candidate_labels,35
5,registry_rows_in_standardised_fundamentals,78
6,registry_unique_labels_in_standardised_fundame...,26
7,registry_unique_concepts_in_standardised_funda...,14
8,standardised_fundamental_rows,659
9,remaining_unmapped_inventory_rows,15320


Block 8 Block 9 feedback diagnostic passed.


## Operating notes

### Scope

Block 8 covers ASX-listed automotive manufacturers, suppliers, battery-material companies and related issuers represented in the point-in-time ETF universe.

### Deterministic extraction

The notebook uses official ASX disclosures, annual reports, PDF text extraction, curated account dictionaries, exact matching, fuzzy matching and accounting validation. It does not perform AI-assisted interpretation.

### Review queues

Ambiguous records are retained rather than forced into the canonical dataset. The Block 8 review outputs are intended for Block 9 and include unresolved account mappings, suspicious values, duplicate facts, weak entity matches and document-classification exceptions.

### Source precedence

Usable structured regulatory fundamentals from upstream blocks remain preferred where they provide sufficient issuer-level coverage. ASX facts are primary or supplementary according to issuer coverage, canonical-concept availability and filing provenance.

### Published contract

All persisted outputs are written to:

```python
data/interim/block_8/
```

The manifest is:

```python
data/interim/block_8/block_8_manifest.json
```

Block 9 consumes the deterministic Australia outputs. Block 10 should consume the quality-controlled outputs produced by Block 9.
